In [1]:
import pandas as pd
import numpy as np
from openai import OpenAI
import json
import re
from difflib import get_close_matches
from rapidfuzz import process, fuzz

In [99]:
df = pd.read_csv('real_estate_ads.csv') # DtypeWarning: Columns (10,26,28,52) have mixed types.
df.head()

# I will fix and convert the columns to their correct type (int, boolean, ...)

C:\Users\Parinaz\AppData\Local\Temp\ipykernel_9020\2019613681.py:1: DtypeWarning: Columns (10,26,28,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('real_estate_ads.csv') # DtypeWarning: Columns (10,26,28,52) have mixed types.


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,rent_value,...,property_type,regular_person_capacity,extra_person_capacity,cost_per_extra_person,rent_price_on_regular_days,rent_price_on_special_days,rent_price_at_weekends,location_latitude,location_longitude,location_radius
0,temporary-rent,villa,karaj,mehrshahr,2024-08-01 00:00:00,مشاور املاک,۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه,NaN,NaN,...,NaN,4.0,6,350000.0,1500000.0,3.500000e+09,3500000.0,35.811684,50.936600,500.0
1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0
2,residential-rent,apartment-rent,tehran,tohid,2024-10-01 00:00:00,NaN,تخلیه پایان ماه,آپارتمان ۳ خوابه ۱۳۲ متر,مقطوع,26000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.703865,51.373459,NaN
3,commercial-rent,office-rent,tehran,elahiyeh,2024-06-01 00:00:00,NaN,فرشته تاپ لوکیشن\n۹۰ متر موقعیت اداری\nیک اتاق...,فرشته ۹۰ متر دفتر کار مدرن موقعیت اداری,مقطوع,95000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01 00:00:00,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [101]:
orig_df = df

### Correcting the types for each column and cleaning (by exploration)

In [49]:
df.info()

# NOTE: rent_to_single (column 10) only has 19 non-null so should be removed

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 60 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   cat2_slug                   1000000 non-null  object 
 1   cat3_slug                   999999 non-null   object 
 2   city_slug                   999998 non-null   object 
 3   neighborhood_slug           437139 non-null   object 
 4   created_at_month            1000000 non-null  object 
 5   user_type                   288882 non-null   object 
 6   description                 1000000 non-null  object 
 7   title                       999946 non-null   object 
 8   rent_mode                   352994 non-null   object 
 9   rent_value                  351322 non-null   float64
 10  rent_to_single              19 non-null       object 
 11  rent_type                   103961 non-null   object 
 12  price_mode                  573606 non-null   object 
 13

In [55]:
def check_for_each_column(column_name):
    print(f'=== {column_name} ===')
    print('Number of null values:', df[column_name].isna().sum())
    print('Value counts:', df[column_name].value_counts(), "\n")

for c in df.columns:
    check_for_each_column(c)

=== cat2_slug ===
Number of null values: 0
Value counts: cat2_slug
residential-sell        558708
residential-rent        276558
commercial-rent          76567
commercial-sell          38861
temporary-rent           29903
real-estate-services     19403
Name: count, dtype: int64 

=== cat3_slug ===
Number of null values: 1
Value counts: cat3_slug
apartment-sell                        303385
apartment-rent                        211880
plot-old                              133570
house-villa-sell                      121753
house-villa-rent                       64678
shop-rent                              45993
shop-sell                              21855
office-rent                            21418
suite-apartment                        16465
presell                                15781
villa                                  12899
industry-agriculture-business-sell     11851
industry-agriculture-business-rent      9155
office-sell                             5155
partnership           

Note1: Dropping `rent_to_single` column

Note2: **String** converting for `cat2_slug`, `cat3_slug`, `city_slug`, `neighborhood_slug`, `description`, `rent_mode`, `price_mode`, `credit_mode`, `deed_type`, `has_warm_water_provider`, `property_type`, 

Note3: Adding different columns for `year_created` and `month_created` based on `created_at_month` (deleting this column)

Note 4: **Are Already float64** and no need to preprocess for columns: rent_value,	price_value, credit_value, transformable_credit, transformed_credit, transformable_rent, transformed_rent	land_size, building_size, regular_person_capacity, cost_per_extra_person, rent_price_on_regular_days, rent_price_on_special_days, rent_price_at_weekends, location_latitude, location_longitude, location_radius

Note5: I will `rent_type` as String but it should be a column named `rent_credit` with boolean values True and False

Note6: **Boolean** for `rent_credit_transform`, `transformable_price`, `has_business_deed`, `has_elevator`, `has_warehouse`, `has_parking`, `is_rebuilt`, `has_water`, `has_electricity`, `has_gas`, `has_security_guard`, `has_barbecue`, `has_pool`, `has_jacuzzi`, `has_sauna`, 

Note7: **`floor`** and **`extra_person_capacity`** has 30+ in it, making it non-integer

Note8: **`rooms_count`** is the same as `floor`, but with the difference that the numbers are یک and ... and there is پنج یا بیشتر which needs to be addressed when I turn it into int.

Note9: `total_floors_count` is **int** but has **unselect** instead of NaN, making the column not be int (maybe a good approach is making these unselect be NaN)

Note10: `unit_per_floor` is the same as Note9 but apart with having **unselect** it also has **more_than_8**

Note11: `has_balcony` has these values: `True, true, False, false, unselect`, so what I need to do is to combine the T/F together, mark the column as boolean and then convert unselect to NaN

Note12: `construction_year` has **persian years** and also has a `قبل از ۱۳۷۰` which should be handled

Note13: ...

Note14: **Is string with unselect** for`has_warm_water_provider` and `has_heating_system`, `has_cooling_system`, `has_restroom`, `building_direction`, `floor_material` and then should turn **`unselect` to NaN** for them.









### Preprocessing and data cleaning based on my gathered notes

In [169]:
df = orig_df

In [170]:
# Note1

df = df.drop(columns='rent_to_single')

In [171]:
# checking before

df['construction_year'].value_counts()

construction_year
۱۴۰۳           116260
۱۳۹۰            59139
۱۴۰۲            58424
۱۴۰۰            53674
۱۳۹۵            53029
۱۳۹۸            38207
۱۳۹۷            36326
۱۳۹۶            35487
۱۴۰۱            35328
۱۳۸۵            34065
۱۳۹۹            29594
۱۳۹۳            29094
۱۳۹۲            26130
۱۳۹۴            26110
۱۳۸۸            24268
۱۳۸۰            23480
قبل از ۱۳۷۰     20637
۱۳۸۹            16755
۱۳۹۱            16316
۱۳۸۷            14136
۱۳۸۶            13468
۱۳۸۳             9894
۱۳۸۴             8494
۱۳۷۵             7247
۱۳۸۲             6965
۱۳۷۱             5531
۱۳۸۱             3590
۱۳۷۸             3025
۱۳۷۹             2415
۱۳۷۷             2117
۱۳۷۲             1914
۱۳۷۳             1827
۱۳۷۶             1593
۱۳۷۴             1289
Name: count, dtype: int64

In [172]:
# mapping Persian digits to ASCII 0–9
_EASTERN_TO_ASCII = str.maketrans(
    '۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩',
    '01234567890123456789'
)

def _parse_construction_year(val):
    if pd.isna(val):
        return np.nan
    s = str(val).translate(_EASTERN_TO_ASCII).strip()

    m = re.search(r'(\d{3,4})', s)
    return int(m.group(1)) if m else np.nan

In [173]:
def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    # ----------------------------
    # Step 1: String conversions + unselect -> NaN
    # ----------------------------
    string_cols = [
        'cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
        'description', 'rent_mode', 'price_mode', 'credit_mode',
        'deed_type', 'has_warm_water_provider', 'property_type',
        'has_heating_system', 'has_cooling_system', 'has_restroom',
        'building_direction', 'floor_material'
    ]
    for col in string_cols:
        if col in df.columns:
            df[col] = df[col].replace('unselect', np.nan).astype(str)

    # ----------------------------
    # Step 2: Extract year/month from created_at_month (Note 3)
    # ----------------------------
    if 'created_at_month' in df.columns:
        # Assuming created_at_month format: YYYY-MM or YYYY/MM
        df['year_created'] = pd.to_datetime(df['created_at_month'], errors='coerce').dt.year
        df['month_created'] = pd.to_datetime(df['created_at_month'], errors='coerce').dt.month
        df = df.drop(columns=['created_at_month'])

    # ----------------------------
    # Step 3: Boolean conversions (Note 6 + Note 11)
    # ----------------------------
    bool_cols = [
        'rent_credit_transform', 'transformable_price', 'has_business_deed',
        'has_elevator', 'has_warehouse', 'has_parking', 'is_rebuilt',
        'has_water', 'has_electricity', 'has_gas', 'has_security_guard',
        'has_barbecue', 'has_pool', 'has_jacuzzi', 'has_sauna'
    ]
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].astype('boolean')

    # has_balcony special handling (True/true/False/false/unselect) and then convereted to Boolean
    if 'has_balcony' in df.columns:
        df['has_balcony'] = df['has_balcony'].replace(
            {'true': True, 'True': True, 'false': False, 'False': False, 'unselect': np.nan}
        ).astype('boolean')

    # ----------------------------
    # Step 4: rent_type -> rent_credit boolean (Note 5), and not turning NaN values to False
    # ----------------------------
    if 'rent_type' in df.columns:
        df['rent_credit'] = df['rent_type'].apply(
            lambda x: np.nan if pd.isna(x) else str(x).strip().lower() == 'rent_credit'
        )
        df = df.drop(columns=['rent_type'])

    # ----------------------------
    # Step 5: Numeric fixes (Notes 7–10, 12)
    # ----------------------------
    # floor & extra_person_capacity: remove 30+ and turning it to 40
    for col in ['floor', 'extra_person_capacity', 'total_floors_count']:
        if col in df.columns:
            df[col] = df[col].replace({'30+': 40})
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    # rooms_count: Persian numbers and "پنج یا بیشتر" with "بدون اتاق"
    persian_num_map = {
        'بدون اتاق': 0,
        'یک': 1,
        'دو': 2,
        'سه': 3,
        'چهار': 4,
        'پنج': 5,
        'پنج یا بیشتر': 6
    }

    if 'rooms_count' in df.columns:
        df['rooms_count'] = df['rooms_count'].replace(persian_num_map)
        df['rooms_count'] = df['rooms_count'].astype('object').replace(persian_num_map)

    # total_floors_count & unit_per_floor: unselect -> NaN, more_than_8 -> 9
    if 'total_floors_count' in df.columns:
        df['total_floors_count'] = df['total_floors_count'].replace('unselect', np.nan)
        df['total_floors_count'] = pd.to_numeric(df['total_floors_count'], errors='coerce').astype('float64')

    if 'unit_per_floor' in df.columns:
        df['unit_per_floor'] = df['unit_per_floor'].replace({'unselect': np.nan, 'more_than_8': 9})
        df['unit_per_floor'] = pd.to_numeric(df['unit_per_floor'], errors='coerce', downcast='integer') # .astype('float64')

    # ----------------------------
    # Step 6: construction_year fix (Note 12)
    # ----------------------------
    if 'construction_year' in df.columns:
        df['construction_year'] = df['construction_year'].replace({'قبل از ۱۳۷۰': 1369})
        df['construction_year'] = df['construction_year'].apply(_parse_construction_year).astype('float64')

    return df

In [174]:
df = preprocess_df(df)
df['construction_year'].value_counts()

C:\Users\Parinaz\AppData\Local\Temp\ipykernel_9020\250705973.py:74: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['rooms_count'] = df['rooms_count'].replace(persian_num_map)
C:\Users\Parinaz\AppData\Local\Temp\ipykernel_9020\250705973.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['rooms_count'] = df['rooms_count'].astype('object').replace(persian_num_map)


construction_year
1403.0    116260
1390.0     59139
1402.0     58424
1400.0     53674
1395.0     53029
1398.0     38207
1397.0     36326
1396.0     35487
1401.0     35328
1385.0     34065
1399.0     29594
1393.0     29094
1392.0     26130
1394.0     26110
1388.0     24268
1380.0     23480
1369.0     20637
1389.0     16755
1391.0     16316
1387.0     14136
1386.0     13468
1383.0      9894
1384.0      8494
1375.0      7247
1382.0      6965
1371.0      5531
1381.0      3590
1378.0      3025
1379.0      2415
1377.0      2117
1372.0      1914
1373.0      1827
1376.0      1593
1374.0      1289
Name: count, dtype: int64

### IMPORTANT NOTE: (for the future)

- For `floor`, `extra_person_capacity`, `total_floors_count` replaced **30+** values with **40** to later address them as values being 30+
- For `unit_per_floor` replaced **more_than_8** to **9**
- For `construction_year` replaced **قبل از 1369** with **1369**

In [176]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 60 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   cat2_slug                   1000000 non-null  object 
 1   cat3_slug                   1000000 non-null  object 
 2   city_slug                   1000000 non-null  object 
 3   neighborhood_slug           1000000 non-null  object 
 4   user_type                   288882 non-null   object 
 5   description                 1000000 non-null  object 
 6   title                       999946 non-null   object 
 7   rent_mode                   1000000 non-null  object 
 8   rent_value                  351322 non-null   float64
 9   price_mode                  1000000 non-null  object 
 10  price_value                 568346 non-null   float64
 11  credit_mode                 1000000 non-null  object 
 12  credit_value                352095 non-null   float64
 13

### Limiting the data frame range for less NaN values

In [ ]:
df.to_csv('preprocessed_df_v0.csv', index=False)

In [193]:
filtered_df = df[
    (df['cat2_slug'] == 'residential-sell') &
    (df['cat3_slug'] == 'apartment-sell')
]

filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 303385 entries, 1 to 999995
Data columns (total 60 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   cat2_slug                   303385 non-null  object 
 1   cat3_slug                   303385 non-null  object 
 2   city_slug                   303385 non-null  object 
 3   neighborhood_slug           303385 non-null  object 
 4   user_type                   110664 non-null  object 
 5   description                 303385 non-null  object 
 6   title                       303372 non-null  object 
 7   rent_mode                   303385 non-null  object 
 8   rent_value                  0 non-null       float64
 9   price_mode                  303385 non-null  object 
 10  price_value                 303380 non-null  float64
 11  credit_mode                 303385 non-null  object 
 12  credit_value                0 non-null       float64
 13  rent_credit_transfo

### Cleaning the filtered_df

In [200]:
# dropping columns that have 0 or 1 non-null values
print("Filered dataframe shape before:", filtered_df.shape)

# drop columns that have 0 or 1 non-null values
cols_to_drop = [col for col in filtered_df.columns if filtered_df[col].dropna().shape[0] <= 1]
filtered_df_cleaned = filtered_df.drop(columns=cols_to_drop)

print("Dropped columns:", cols_to_drop)
print("New dataframe shape:", filtered_df_cleaned.shape)

Filered dataframe shape before: (303385, 60)
Dropped columns: ['rent_value', 'credit_value', 'rent_credit_transform', 'transformable_price', 'transformable_credit', 'transformed_credit', 'transformable_rent', 'transformed_rent', 'land_size', 'has_business_deed', 'has_water', 'has_electricity', 'has_gas', 'has_security_guard', 'has_barbecue', 'has_pool', 'has_jacuzzi', 'has_sauna', 'regular_person_capacity', 'extra_person_capacity', 'cost_per_extra_person', 'rent_price_on_regular_days', 'rent_price_on_special_days', 'rent_price_at_weekends', 'rent_credit']
New dataframe shape: (303385, 35)


In [201]:
filtered_df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 303385 entries, 1 to 999995
Data columns (total 35 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   cat2_slug                303385 non-null  object 
 1   cat3_slug                303385 non-null  object 
 2   city_slug                303385 non-null  object 
 3   neighborhood_slug        303385 non-null  object 
 4   user_type                110664 non-null  object 
 5   description              303385 non-null  object 
 6   title                    303372 non-null  object 
 7   rent_mode                303385 non-null  object 
 8   price_mode               303385 non-null  object 
 9   price_value              303380 non-null  float64
 10  credit_mode              303385 non-null  object 
 11  building_size            303380 non-null  float64
 12  deed_type                303385 non-null  object 
 13  floor                    303377 non-null  Int64  
 14  rooms_cou

In [203]:
filtered_df_cleaned.to_csv('filtered_df_cleaned.csv', index=False)

In [204]:
df = filtered_df_cleaned

### Mixing `title` and `description` columns into a single one for future ML analysis

```python
df['description_title'] = df['title'].astype(str) + " " + df['description'].astype(str)
```

Commented it so it would be used when I want to extract information from rows using ML

# Extracting column names and their values from user input

In [179]:
for c in df.columns:
    check_for_each_column(c)

=== cat2_slug ===
Number of null values: 0
Value counts: cat2_slug
residential-sell        558708
residential-rent        276558
commercial-rent          76567
commercial-sell          38861
temporary-rent           29903
real-estate-services     19403
Name: count, dtype: int64 

=== cat3_slug ===
Number of null values: 0
Value counts: cat3_slug
apartment-sell                        303385
apartment-rent                        211880
plot-old                              133570
house-villa-sell                      121753
house-villa-rent                       64678
shop-rent                              45993
shop-sell                              21855
office-rent                            21418
suite-apartment                        16465
presell                                15781
villa                                  12899
industry-agriculture-business-sell     11851
industry-agriculture-business-rent      9155
office-sell                             5155
partnership           

In [18]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
)

COLUMNS = df.columns

COLUMNS

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug', 'user_type',
       'description', 'title', 'rent_mode', 'price_mode', 'price_value',
       'credit_mode', 'building_size', 'deed_type', 'floor', 'rooms_count',
       'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator',
       'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt',
       'has_warm_water_provider', 'has_heating_system', 'has_cooling_system',
       'has_restroom', 'building_direction', 'floor_material',
       'location_latitude', 'location_longitude', 'location_radius',
       'year_created', 'month_created'],
      dtype='object')

In [181]:
def parse_user_query(user_text):
    prompt = f"""
        You are a JSON field extractor.
        The input will be a Persian text describing a property search.
        Your task:
        - Read the Persian text carefully.
        - Identify which fields from this dataset columns list the user mentioned: {COLUMNS}
        - Only include columns that are explicitly mentioned or implied.
        - Extract the exact value for each mentioned column from the sentence.
        - If a value is numeric, return it as a number (not a string).
        - Output valid JSON only, with keys exactly as in the list and values.
        - Omit any field not mentioned.
        - If a number is given in Persian words (یک, دو, سه), convert to integer.
        - Ignore irrelevant text.
        
        Example:
        User: "خونه ۳ خواب در قلهک تهران، قیمت تا ۵۰ میلیارد، ۱۰۰ متر"
        Output:
        {{
          "city_slug": "tehran",
          "neighborhood_slug": "gholhak",
          "rooms_count": 3,
          "price_value": 50000000000,
          "building_size": 100
        }}
        """

    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You output JSON only."},
            {"role": "user", "content": prompt + "\nUser: " + user_text}
        ]
    )
    
    content = completion.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {"error": "Invalid JSON returned", "raw_output": content}

### Testing

In [183]:
df.iloc[10,:]

cat2_slug                                                      residential-sell
cat3_slug                                                        apartment-sell
city_slug                                                           pardis-city
neighborhood_slug                                                           nan
user_type                                                                   NaN
description                   سردرب عالی \nویو عالی بدون مشرف \nنصبیات نصب ش...
title                                         فاز ۱۱ زون ۲ ساخت شرکت کوزو ترکیه
rent_mode                                                                   nan
rent_value                                                                  NaN
price_mode                                                                مقطوع
price_value                                                        2600000000.0
credit_mode                                                                 nan
credit_value                            

In [184]:
custom_user_input1 = "آپارتمان ۲ خوابه، ۸۷ متر، طبقه چهارم، فاز ۱۱ زون ۲ پردیس، ساخت شرکت کوزو ترکیه، قیمت مقطوع ۲ میلیارد و ۶۰۰ میلیون، شمالی، کف سرامیک، بالکن‌دار، پارکینگ، آسانسور، انباری."
custom_user_input2 = "فروش فوری آپارتمان ۸۷ متری در پردیس، طبقه ۴ از ۱۴، دو خواب، ساخت سال ۱۴۰۳، سند شخصی، قیمت ۲.۶ میلیارد."
custom_user_input3 = "آپارتمان در فاز ۱۱ پردیس، ۲ خوابه، کف سرامیک، شمالی، ویو باز، آسانسور، پارکینگ، انباری، ۲ میلیارد و ۶۰۰."
custom_user_input4 = "خرید آپارتمان کوزو ترکیه‌ای، ۸۷ متر، ۲ خوابه، پردیس فاز ۱۱، قیمت مقطوع ۲ میلیارد و ۶۰۰ میلیون، سال ساخت ۱۴۰۳."
custom_user_input5 = "فروش آپارتمان در پردیس، زون ۲، ۸۷ متر، دو اتاق، طبقه ۴، آسانسور و پارکینگ دارد، قیمت ۲ میلیارد و ششصد."
custom_user_input6 = "آپارتمان نوساز ۱۴۰۳ پردیس فاز ۱۱، ۲ خواب، کف سرامیک، بالکن، شمالی، پارکینگ و آسانسور کامل، قیمت ۲۶۰۰."
custom_user_input7 = "پردیس، فاز ۱۱، زون ۲، آپارتمان ۸۷ متری، ۲ خواب، ۴ طبقه، آسانسور، پارکینگ، انباری، ۲ میلیارد و ۶۰۰ میلیون."
custom_user_input8 = "واحد فروشی پردیس فاز ۱۱ کوزو ترکیه‌ای، ۲ خوابه، ۸۷ متر، سند تک‌برگ، کف سرامیک، شمالی، ۲ میلیارد و ششصد."
custom_user_input9 = "فروش آپارتمان ۸۷ متر، طبقه چهارم از ۱۴، فاز ۱۱ پردیس، ۲ خواب، ویوی عالی، کف سرامیک، سال ساخت ۱۴۰۳، قیمت مقطوع ۲.۶ میلیارد."
custom_user_input10 = "آپارتمان نوساز در پردیس، شمالی، بالکن‌دار، ۲ خواب، ۸۷ متر، پارکینگ و انباری، آسانسور، قیمت مقطوع ۲۶۰۰."
custom_user_input11 = "فروش آپارتمان دو خوابه ۸۷ متری، فاز ۱۱ پردیس، طبقه ۴، کف سرامیک، قیمت ۲ میلیارد و ۶۰۰ میلیون."
custom_user_input12 = "پردیس، فاز ۱۱، آپارتمان ۸۷ متر، کوزو ترکیه، ساخت ۱۴۰۳، ۲ خواب، شمالی، ۲ میلیارد و ششصد، آسانسور و پارکینگ دارد."
custom_user_input13 = "آپارتمان با ۲ خواب، ۸۷ متر، کف سرامیک، شمالی، پردیس زون ۲، طبقه چهارم، پارکینگ و آسانسور کامل، قیمت ۲۶۰۰."
custom_user_input14 = "فروش آپارتمان ۸۷ متری، فاز ۱۱ پردیس، سال ساخت ۱۴۰۳، ۲ خواب، طبقه چهار، سند رسمی، قیمت مقطوع ۲ میلیارد و ۶۰۰ میلیون."
custom_user_input15 = "آپارتمان فروشی پردیس، کوزو ترکیه‌ای، نوساز، ۸۷ متر، ۲ خوابه، شمالی، کف سرامیک، آسانسور، پارکینگ، انباری، قیمت ۲.۶ میلیارد."

In [185]:
# user_input = input("Enter user text: ") # taking input from the user
user_input = custom_user_input1
extracted = parse_user_query(user_input)
print(extracted)

{'rooms_count': 2, 'building_size': 87, 'floor': 4, 'neighborhood_slug': 'faza-11-zone-2-pardis', 'price_value': 2600000000, 'has_balcony': True, 'has_parking': True, 'has_elevator': True, 'has_warehouse': True, 'building_direction': 'northern', 'floor_material': 'ceramic'}


In [186]:
def testing_input_loop():
    while True:
        user_input = input("\nEnter user text: ") # taking input from the user
        if user_input == 'exit':
            break
        extracted = parse_user_query(user_input)
        print(extracted)

testing_input_loop()

Enter user text:  سلام در این تکست من اطلاعاتی وارد نمیکنم تا چک کنم و 2 ببینم که 45666 مدل ستونی رو  844 تشخیص میده یا نه


{'rooms_count': 2, 'price_value': 45666, 'rent_value': 844}


Enter user text:  مثل اینکه 2 تا باید بیشتر روی پرامپ کار 8373 بکنم تا 24 چیزی رو اشتباه تشخیص نده که 10000 نیست


{'rent_value': 2, 'price_value': 10000}


Enter user text:  سلام


...

Enter user text:  یک تکست بدون داشتن هیچ عددی مثلا فقط عدد دو رو مینویسم و چک کنم ببینم چی جواب میده


{'rooms_count': 2}


Enter user text:  پس احتمالا با وارد کردن اعداد مشکل داره و برای تکست خالی اگر سایز خونه و سال و تعداد اتاق و هیچ عددی رو وارد نکنم احتمالا اشتباه تشخیص نمیده


{'error': 'Invalid JSON returned', 'raw_output': '```json\n{}\n```'}


Enter user text:  exit


### Testing all of the custom inputs

In [187]:
custom_inputs = [
    custom_user_input1, custom_user_input2, custom_user_input3, custom_user_input4, custom_user_input5,
    custom_user_input6, custom_user_input7, custom_user_input8, custom_user_input9, custom_user_input10,
    custom_user_input11, custom_user_input12, custom_user_input13, custom_user_input14, custom_user_input15
]

for i, user_input in enumerate(custom_inputs, start=1):
    print("=" * 60)
    print(f"Test #{i}\n")
    # print("-" * 60)
    print(f"User Input:\n{user_input}\n")
    
    extracted = parse_user_query(user_input)
    print("Extracted Data:")
    print(extracted)
    print("=" * 60 + "\n")

Test #1
------------------------------------------------------------
User Input:
آپارتمان ۲ خوابه، ۸۷ متر، طبقه چهارم، فاز ۱۱ زون ۲ پردیس، ساخت شرکت کوزو ترکیه، قیمت مقطوع ۲ میلیارد و ۶۰۰ میلیون، شمالی، کف سرامیک، بالکن‌دار، پارکینگ، آسانسور، انباری.

Extracted Data:
{'rooms_count': 2, 'building_size': 87, 'floor': 4, 'neighborhood_slug': 'faza-11-zone-2-pardis', 'price_value': 2600000000, 'has_balcony': True, 'has_parking': True, 'has_elevator': True, 'has_warehouse': True, 'building_direction': 'northern', 'floor_material': 'ceramic'}

Test #2
------------------------------------------------------------
User Input:
فروش فوری آپارتمان ۸۷ متری در پردیس، طبقه ۴ از ۱۴، دو خواب، ساخت سال ۱۴۰۳، سند شخصی، قیمت ۲.۶ میلیارد.

Extracted Data:
{'city_slug': 'pardis', 'floor': 4, 'total_floors_count': 14, 'rooms_count': 2, 'construction_year': 1403, 'deed_type': 'شخصی', 'price_value': 2600000000, 'building_size': 87}

Test #3
------------------------------------------------------------
User Inpu

More Testing for finding weaknesses

In [189]:
testing_input_loop()


Enter user text:   سلام من یک آپارتمان میخوام که آسانسورش نباشه و ساختمان دارای 11 تا اتاق باشه. و یک باکنی باشه که بزرگ بناش 


{'has_elevator': False, 'rooms_count': 11, 'has_balcony': True}



Enter user text:  یک خونه توی منطقه فلسطین توی مشهد میخوام که سند رسمی داشته باشه و ارتفاع 100 داشته باشه و 2 تا اتاق باشه با یک اتاق بیشتر برای یک فرد دیگه که اون اتاق بالکنداره


{'city_slug': 'mashhad', 'neighborhood_slug': 'falestin', 'has_business_deed': True, 'building_size': 100, 'rooms_count': 2, 'regular_person_capacity': 1, 'extra_person_capacity': 1}



Enter user text:  exit


**NOTE**: I should limit the column searches for example `extra_person_capacity` and all of that.

**NOTE**: And then I should find out the range of each column to give it to the LLM

### Limiting the columns considered by the LLM

**THE COLUMNS:**

**The main ones to put** (from my pov):
- `cat3_slug`
- `city_slug`
- `neighborhood_slug`
- `price_value` (*range*)
- `land_size` (*range*)
- `buiding_size` (*range*)
- `rooms_count`

**To also put:** `cat2_slug`, `floor`, `has_balcony`, `has_elevator`, `has_warehouse`, `has_parking`, `has_water`, `has_warm_water_provider`, `construction_year`, `deed_type`, `has_electricity`, `has_gas`, `has_heating_system`, `has_cooling system`, `has_restroom`, `has_security_guard`, `has_barbecue`, `building_direction` (**maybe good**), `has_pool`, `has_jacuzzi`, `has_sauna`, `floor_material`, `property_type` (beach, jungle or ...)

**Can also be put**: `has_business_deed`, `total_floors_count`, `unit_per_floor`, `is_rebuild`

**Sure** to **not** put:
- `user_type` (شخصی or مشاور املاک doesn't matter)
- `description`
- `title`
- `location_latitude` (**may still use for geo**) -> It's not like a user is going to specify latitude and longitude
- `location_radius` (**may still use for geo**)
- `month_created` because no one generally cares about the month in which it was built

In [190]:
df.columns

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug', 'user_type',
       'description', 'title', 'rent_mode', 'rent_value', 'price_mode',
       'price_value', 'credit_mode', 'credit_value', 'rent_credit_transform',
       'transformable_price', 'transformable_credit', 'transformed_credit',
       'transformable_rent', 'transformed_rent', 'land_size', 'building_size',
       'deed_type', 'has_business_deed', 'floor', 'rooms_count',
       'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator',
       'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt',
       'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas',
       'has_heating_system', 'has_cooling_system', 'has_restroom',
       'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool',
       'has_jacuzzi', 'has_sauna', 'floor_material', 'property_type',
       'regular_person_capacity', 'extra_person_capacity',
       'cost_per_extra_person', 'rent_price_on

Columns to put if the data was unfiltered on `cat2_slug` and `cat3_slug`:

```python
COLUMNS_TO_USE_UNFILTERED = [
    "cat3_slug", "city_slug", "neighborhood_slug", "price_value", "building_size", "rooms_count", "land_size",
     # Secondary
     "cat2_slug", "floor", "has_balcony", "has_elevator", "has_warehouse", "has_parking",
     "has_water", "has_warm_water_provider", "construction_year", "deed_type", "has_electricity",
     "has_gas", "has_heating_system", "has_cooling_system", "has_restroom", "has_security_guard",
     "has_barbecue", "building_direction", "has_pool", "has_jacuzzi", "has_sauna", "floor_material", "property_type",
     # Optional / rarely used
     "has_business_deed", "total_floors_count", "unit_per_floor", "is_rebuilt"
]
```

Notes:
- `has_cooling_system` and `has_restroom` and `floor_material` have value **nan** amongst their other values
- `property_type` only has a value named **nan**, I should drop it

In [211]:
columns_to_fix = ["has_cooling_system", "has_restroom", "floor_material"]
print('Sum of null:', df[columns_to_fix].isna().sum())

Sum of null: has_cooling_system    0
has_restroom          0
floor_material        0
dtype: int64


In [213]:
# replacing string 'nan' with actual np.nan
columns_to_fix = ["has_cooling_system", "has_restroom", "floor_material"]
for col in columns_to_fix:
    df[col] = df[col].replace("nan", np.nan)

# dropping the 'property_type' column because it only contains np.nan
df = df.drop(columns=["property_type"])

print(df[columns_to_fix].isna().sum())
print("Columns left:", df.columns)

has_cooling_system    138667
has_restroom          114717
floor_material        110994
dtype: int64
Columns left: Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug', 'user_type',
       'description', 'title', 'rent_mode', 'price_mode', 'price_value',
       'credit_mode', 'building_size', 'deed_type', 'floor', 'rooms_count',
       'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator',
       'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt',
       'has_warm_water_provider', 'has_heating_system', 'has_cooling_system',
       'has_restroom', 'building_direction', 'floor_material',
       'location_latitude', 'location_longitude', 'location_radius',
       'year_created', 'month_created'],
      dtype='object')


In [214]:
# saving the file with all the changes
df.to_csv('filtered_df_cleaned_v1', index=False)

**Added** to prompt *Only assign a numeric value to a column if the text clearly states that number in direct connection with that column’s meaning* to the prompt

And changed COLUMNS to COLUMNS_TO_USE

In [216]:
COLUMNS_TO_USE = [
    # Main
    "city_slug",
    "neighborhood_slug",
    "price_value",
    "building_size",
    "rooms_count",
    
    # Secondary
    "floor",
    "has_balcony",
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_warm_water_provider",
    "construction_year",
    "deed_type",
    "has_heating_system",
    "has_cooling_system",
    "has_restroom",
    "building_direction",
    "floor_material",
    
    # Optional / rarely used
    "total_floors_count",
    "unit_per_floor",
    "is_rebuilt"
]


def parse_user_query(user_text):
    prompt = f"""
        You are a JSON field extractor.
        The input will be a Persian text describing a property search.
        Your task:
        - Read the Persian text carefully.
        - Identify which fields from this dataset columns list the user mentioned: {COLUMNS_TO_USE}
        - Only assign a numeric value to a column if the text clearly states that number in direct connection with that column’s meaning
        - Only include columns that are explicitly mentioned or implied.
        - Extract the exact value for each mentioned column from the sentence.
        - If a value is numeric, return it as a number (not a string).
        - Output valid JSON only, with keys exactly as in the list and values.
        - Omit any field not mentioned.
        - If a number is given in Persian words (یک, دو, سه), convert to integer.
        - Ignore irrelevant text.
        
        Example:
        User: "یه خونه سه خوابه تو قلهک تهران می‌خوام، قیمتش تا ۵۰ میلیارد، صد متره، بالکن و آسانسور داره، پارکینگ هم داره، طبقه ۲، سند تک‌برگه، آبگرم‌کن داره، سال ساخت ۱۴۰۰، شوفاژ و کولر داره، دستشویی ایرانیه، به سمت جنوبه، کفش موزاییکه، نوع ملک معمولیه، کل ساختمان ۴ طبقه و هر طبقه ۲ واحد، انبار نداره و بازسازی نشده"
        Output:
        {{
          "city_slug": "tehran",
          "neighborhood_slug": "gholhak",
          "rooms_count": 3,
          "price_value": 50000000000,
          "building_size": 100,
          "has_balcony": True,
          "has_elevator": True,
          "has_parking": True,
          "floor": 2,
          "deed_type": "single_page",
          "has_warm_water_provider": "water_heater",
          "construction_year": 1400,
          "has_heating_system": "shoofaj",
          "has_cooling_system": "air_conditioner",
          "has_restroom": "squat_seat",
          "building_direction": "south",
          "floor_material": "mosaic",
          "total_floors_count": 5,
          "unit_per_floor": 2,
          "is_rebuilt": False,
          "has_warehouse": False,
          "total_floors_count": 4
        }}
        """

    deed_type, has_warm_water_provider, has_heating_system, has_cooling_system, has_restoroom, floor_material

    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You output JSON only."},
            {"role": "user", "content": prompt + "\nUser: " + user_text}
        ]
    )
    
    content = completion.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        # return {"error": "Invalid JSON returned", "raw_output": content}
        return {}

Working on **Example sentence** for the prompt

In [217]:
sample1 = "یه آپارتمان ۲ خوابه تو مهرشهر تهران می‌خوام، قیمتش تا ۳۰ میلیارد، ۸۰ متره، بالکن و آسانسور داره، طبقه ۳، سال ساخت ۱۳۹۵، هیتر داره و به سمت شماله"

How the output should be for this sentence:

```python
Output:
{
  "city_slug": "tehran",
  "neighborhood_slug": "mehrshahr",
  "rooms_count": 2,
  "price_value": 30000000000,
  "building_size": 80,
  "has_balcony": True,
  "has_elevator": True,
  "floor": 3,
  "construction_year": 1395,
  "has_heating_system": "heater",
  "building_direction": "north"
}

```

In [218]:
# The output matches exactly!

user_input = sample1
extracted = parse_user_query(user_input)
print(extracted)

{'city_slug': 'tehran', 'neighborhood_slug': 'mehrshahr', 'rooms_count': 2, 'price_value': 30000000000, 'building_size': 80, 'has_balcony': True, 'has_elevator': True, 'floor': 3, 'construction_year': 1395, 'has_heating_system': 'heater', 'building_direction': 'north'}


In [219]:
sample2 = "سلام در این تکست من اطلاعاتی وارد نمیکنم تا چک کنم و 2 ببینم که 45666 مدل ستونی رو  844 تشخیص میده یا نه"
user_input = sample2
extracted = parse_user_query(user_input)
print(extracted)

{'floor': 2}


### Specifying values for each column

These columns should have **their values mentioned to the LLM**:
deed_type, has_warm_water_provider, has_heating_system, has_cooling_system, has_restoroom, floor_material

**Added** the below block to the prompt:

```bash
Columns and their allowed values:

- deed_type: ["single_page", "written_agreement", "multi_page", "other"]
- has_warm_water_provider: ["water_heater", "package", "powerhouse"]
- has_heating_system: ["shoofaj", "heater", "duct_split", "floor_heating", "fan_coil", "split", "fireplace"]
- has_cooling_system: ["air_conditioner", "water_cooler", "duct_split", "split", "fan_coil"]
- has_restroom: ["squat_seat", "squat", "seat"]
- floor_material: ["mosaic", "ceramic", "stone", "wood_parquet", "laminate_parquet", "carpet", "floor_covering"]
```

In [230]:
#  Having "nan" in their values: columns has_warm_water_provider and deed_type -> converting to np.nan

columns_to_fix2 = ["has_warm_water_provider", "deed_type"]
for col in columns_to_fix2:
    df[col] = df[col].replace("nan", np.nan)

In [231]:
def parse_user_query(user_text):
    prompt = f"""
        You are a JSON field extractor.
        The input will be a Persian text describing a property search.
        Your task:
        - Read the Persian text carefully.
        - Identify which fields from this dataset columns list the user mentioned: {COLUMNS_TO_USE}
        - Only assign a numeric value to a column if the text clearly states that number in direct connection with that column’s meaning
        - Only include columns that are explicitly mentioned or implied.
        - Extract the exact value for each mentioned column from the sentence.
        - If a value is numeric, return it as a number (not a string).
        - Output valid JSON only, with keys exactly as in the list and values.
        - Omit any field not mentioned.
        - If a number is given in Persian words (یک, دو, سه), convert to integer.
        - Ignore irrelevant text.

        Some of the columns and their allowed values:
        - deed_type: ["single_page", "written_agreement", "multi_page", "other"]
        - has_warm_water_provider: ["water_heater", "package", "powerhouse"]
        - has_heating_system: ["shoofaj", "heater", "duct_split", "floor_heating", "fan_coil", "split", "fireplace"]
        - has_cooling_system: ["air_conditioner", "water_cooler", "duct_split", "split", "fan_coil"]
        - has_restroom: ["squat_seat", "squat", "seat"]
        - floor_material: ["mosaic", "ceramic", "stone", "wood_parquet", "laminate_parquet", "carpet", "floor_covering"
        
        Example:
        User: "یه خونه سه خوابه تو قلهک تهران می‌خوام، قیمتش تا ۵۰ میلیارد، صد متره، بالکن و آسانسور داره، پارکینگ هم داره، طبقه ۲، سند تک‌برگه، آبگرم‌کن داره، سال ساخت ۱۴۰۰، شوفاژ و کولر داره، دستشویی ایرانیه، به سمت جنوبه، کفش موزاییکه، نوع ملک معمولیه، کل ساختمان ۴ طبقه و هر طبقه ۲ واحد، انبار نداره و بازسازی نشده"
        Output:
        {{
          "city_slug": "tehran",
          "neighborhood_slug": "gholhak",
          "rooms_count": 3,
          "price_value": 50000000000,
          "building_size": 100,
          "has_balcony": True,
          "has_elevator": True,
          "has_parking": True,
          "floor": 2,
          "deed_type": "single_page",
          "has_warm_water_provider": "water_heater",
          "construction_year": 1400,
          "has_heating_system": "shoofaj",
          "has_cooling_system": "air_conditioner",
          "has_restroom": "squat_seat",
          "building_direction": "south",
          "floor_material": "mosaic",
          "total_floors_count": 5,
          "unit_per_floor": 2,
          "is_rebuilt": False,
          "has_warehouse": False,
          "total_floors_count": 4
        }}
        """

    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You output JSON only."},
            {"role": "user", "content": prompt + "\nUser: " + user_text}
        ]
    )
    
    content = completion.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        # return {"error": "Invalid JSON returned", "raw_output": content}
        return {"Empty"}

In [232]:
sample3 = "یه آپارتمان ۳ خوابه تو راهنمایی مشهد می‌خوام، قیمتش تا ۵۰ میلیارد، ۱۲۰ متره، بالکن داره و آسانسور هم هست، طبقه ۲، سال ساخت ۱۴۰۰، سندش تک‌برگی، آبگرمکن داره، شوفاژ و کولر گازی، دستشویی ایرانی، کفش موزاییکه، جهت ساختمان جنوب، پارکینگ داره"
user_input = sample3
extracted = parse_user_query(user_input)
print(extracted)

{'city_slug': 'mashhad', 'neighborhood_slug': 'rahnamaei', 'rooms_count': 3, 'price_value': 50000000000, 'building_size': 120, 'has_balcony': True, 'has_elevator': True, 'floor': 2, 'deed_type': 'single_page', 'has_warm_water_provider': 'water_heater', 'construction_year': 1400, 'has_heating_system': 'shoofaj', 'has_cooling_system': 'split', 'has_restroom': 'squat_seat', 'building_direction': 'south', 'floor_material': 'mosaic', 'has_parking': True}


Expected output:

```python
{
  "city_slug": "mashhad",
  "neighborhood_slug": "rahnamaei",
  "rooms_count": 3,
  "price_value": 50000000000,
  "building_size": 120,
  "has_balcony": True,
  "has_elevator": True,
  "floor": 2,
  "construction_year": 1400,
  "deed_type": "single_page",
  "has_warm_water_provider": "water_heater",
  "has_heating_system": "shoofaj",
  "has_cooling_system": "air_conditioner",
  "has_restroom": "squat_seat",
  "floor_material": "mosaic",
  "building_direction": "south",
  "has_parking": True
}
```

**Mistakes**: The only mistake was instead of `has_cooling_system: "air_conditioner` it wrote `has_cooling_system: split`. Which is natural and even humans can make a mistake of.

#### Scenario 1 from the project description

In [234]:
scenario1 = "سلام. من دنبال یک خونه ویلایی توی هاشمیه مشهد برای اجاره هستم. حداقل دو خواب داشته باشه و پول پیشش بیشتر از ۸۰۰ میلیون نباشه."
user_input = scenario1
extracted = parse_user_query(user_input)
print(extracted)

{'city_slug': 'mashhad', 'neighborhood_slug': 'hashemieh', 'rooms_count': 2, 'price_value': 800000000}


**Added to the prompt**: The user asked for price_value less than 800000000 so I thought that instead of `'price_value': 800000000`, the LLM would just say `'max_price_value': 800000000`. Adding a **max_** or **min_** can help me handle this later in the code.


In [14]:
def parse_user_query(user_text):
    prompt = f"""
        You are a JSON field extractor.
        The input will be a Persian text describing a property search.
        Your task:
        - Read the Persian text carefully.
        - Identify which fields from this dataset columns list the user mentioned: {COLUMNS_TO_USE}
        - Only assign a numeric value to a column if the text clearly states that number in direct connection with that column’s meaning
        - Only include columns that are explicitly mentioned or implied.
        - Extract the exact value for each mentioned column from the sentence.
        - If a value is numeric, return it as a number (not a string).
        - If the user specifies a range like "حداقل", "حداکثر", "کمتر از", "بیشتر از", then instead of the exact column name, use "min_" or "max_" prefix. 
          Examples:
            - "حداقل ۸۰۰ میلیون" → "min_price_value": 800000000
            - "تا ۸۰۰ میلیون" → "max_price_value": 800000000
        
        - Output valid JSON only, with keys exactly as in the list and values.
        - Omit any field not mentioned.
        - If a number is given in Persian words (یک, دو, سه), convert to integer.
        - Ignore irrelevant text.

        Some of the columns and their allowed values:
        - deed_type: ["single_page", "written_agreement", "multi_page", "other"]
        - has_warm_water_provider: ["water_heater", "package", "powerhouse"]
        - has_heating_system: ["shoofaj", "heater", "duct_split", "floor_heating", "fan_coil", "split", "fireplace"]
        - has_cooling_system: ["air_conditioner", "water_cooler", "duct_split", "split", "fan_coil"]
        - has_restroom: ["squat_seat", "squat", "seat"]
        - floor_material: ["mosaic", "ceramic", "stone", "wood_parquet", "laminate_parquet", "carpet", "floor_covering"
        
        Example:
        User: "یه خونه سه خوابه تو قلهک تهران می‌خوام، قیمتش تا ۵۰ میلیارد، صد متره، بالکن و آسانسور داره، پارکینگ هم داره، طبقه ۲، سند تک‌برگه، آبگرم‌کن داره، سال ساخت ۱۴۰۰، شوفاژ و کولر داره، دستشویی ایرانیه، به سمت جنوبه، کفش موزاییکه، نوع ملک معمولیه، کل ساختمان ۴ طبقه و هر طبقه ۲ واحد، انبار نداره و بازسازی نشده"
        Output:
        {{
          "city_slug": "tehran",
          "neighborhood_slug": "gholhak",
          "rooms_count": 3,
          "price_value": 50000000000,
          "building_size": 100,
          "has_balcony": True,
          "has_elevator": True,
          "has_parking": True,
          "floor": 2,
          "deed_type": "single_page",
          "has_warm_water_provider": "water_heater",
          "construction_year": 1400,
          "has_heating_system": "shoofaj",
          "has_cooling_system": "air_conditioner",
          "has_restroom": "squat_seat",
          "building_direction": "south",
          "floor_material": "mosaic",
          "total_floors_count": 5,
          "unit_per_floor": 2,
          "is_rebuilt": False,
          "has_warehouse": False,
          "total_floors_count": 4
        }}
        """

    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You output JSON only."},
            {"role": "user", "content": prompt + "\nUser: " + user_text}
        ]
    )
    
    content = completion.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {"Empty"}

In [236]:
# Fixed!

scenario1_again = "سلام. من دنبال یک خونه ویلایی توی هاشمیه مشهد برای اجاره هستم. حداقل دو خواب داشته باشه و پول پیشش بیشتر از ۸۰۰ میلیون نباشه."
user_input = scenario1_again
extracted = parse_user_query(user_input)
print(extracted)

{'city_slug': 'mashhad', 'neighborhood_slug': 'hashemieh', 'min_rooms_count': 2, 'max_price_value': 800000000}


In [237]:
df.to_csv('filtered_df_cleaned_v2.csv', index=False)

In [238]:
scenario4 = "سلام. من دنبال خرید خونه توی کالیفرنیا هستم"
user_input = scenario4
extracted = parse_user_query(user_input)
print(extracted)

# California will be handled when checking the ranges of each of the columns

{'city_slug': 'california'}


## Validation System: Checking the ranges of the user's input text

In [252]:
df[COLUMNS_TO_USE].info()

<class 'pandas.core.frame.DataFrame'>
Index: 303385 entries, 1 to 999995
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   city_slug                303385 non-null  object 
 1   neighborhood_slug        303385 non-null  object 
 2   price_value              303380 non-null  float64
 3   building_size            303380 non-null  float64
 4   rooms_count              303379 non-null  float64
 5   floor                    303377 non-null  Int64  
 6   has_balcony              194485 non-null  boolean
 7   has_elevator             303378 non-null  boolean
 8   has_warehouse            303378 non-null  boolean
 9   has_parking              303379 non-null  boolean
 10  has_warm_water_provider  176898 non-null  object 
 11  construction_year        303380 non-null  float64
 12  deed_type                185556 non-null  object 
 13  has_heating_system       303385 non-null  object 
 14  has_cooli

In [10]:
def validate_user_input(user_json, df):
    validated = {}

    for key, value in user_json.items():
        # handling min_/max_ prefixes
        if key.startswith("min_"):
            col = key[4:]
            min_val, max_val = df[col].min(), df[col].max()
            if value < min_val:
                validated[key] = "❌out of range"
            elif value > max_val:
                validated[key] = max_val
            else:
                validated[key] = value
            continue
        elif key.startswith("max_"):
            col = key[4:]
            min_val, max_val = df[col].min(), df[col].max()
            if value > max_val:
                validated[key] = "❌out of range"
            elif value < min_val:
                validated[key] = min_val
            else:
                validated[key] = value
            continue

        col_dtype = df[key].dtype

        # numerical columns
        if pd.api.types.is_numeric_dtype(col_dtype):
            min_val, max_val = df[key].min(), df[key].max()
            if value < min_val or value > max_val:
                validated[key] = "❌out of range"
            else:
                validated[key] = value

        # boolean columns (including nullable boolean)
        elif pd.api.types.is_bool_dtype(col_dtype) or pd.api.types.is_integer_dtype(col_dtype) and df[key].dropna().isin([0,1]).all():
            if value not in [True, False]:
                validated[key] = "❌out of range"
            else:
                validated[key] = value

        # categorical/string columns
        else:
            valid_values = df[key].dropna().unique().tolist()
            if value in valid_values:
                validated[key] = value
            else:
                matches = get_close_matches(value, valid_values, n=1, cutoff=0.7)
                if matches:
                    validated[key] = matches[0]
                else:
                    validated[key] = "❌out of range"

    return validated


def range_test(user_input):
    print('Testing for:', user_input, '\n------------------------------------------')
    
    validated_input = validate_user_input(user_input, df)
    # print(validated_input)
    return validated_input

In [279]:
range_test(extracted)

Testing for: {'city_slug': 'california'} 
------------------------------------------


{'city_slug': '❌out of range'}

In [280]:
sample_range1 = {
    "city_slug": "mashad",  # typo intended
    "neighborhood_slug": "hashemieh",
    "min_rooms_count": 2,
    "max_price_value": 800000000,
    "has_balcony": True
}
range_test(sample_range1)

Testing for: {'city_slug': 'mashad', 'neighborhood_slug': 'hashemieh', 'min_rooms_count': 2, 'max_price_value': 800000000, 'has_balcony': True} 
------------------------------------------


{'city_slug': 'mashhad',
 'neighborhood_slug': 'hashemieh',
 'min_rooms_count': 2,
 'max_price_value': 800000000,
 'has_balcony': True}

In [277]:
sample_range1 = {
    "city_slug": "mashad", # typo
    "neighborhood_slug": "hashemieh",
    "min_rooms_count": 2,
    "max_price_value": 999999999999999, # a very large number
    "has_balcony": True
}
oor1 = range_test(sample_range1)
oor1

Testing for: {'city_slug': 'mashad', 'neighborhood_slug': 'hashemieh', 'min_rooms_count': 2, 'max_price_value': 999999999999999, 'has_balcony': True} 
------------------------------------------
{'city_slug': 'mashhad', 'neighborhood_slug': 'hashemieh', 'min_rooms_count': 2, 'max_price_value': '❌out of range', 'has_balcony': True}


{'city_slug': 'mashhad',
 'neighborhood_slug': 'hashemieh',
 'min_rooms_count': 2,
 'max_price_value': '❌out of range',
 'has_balcony': True}

### Notifying the user the "out of range" message

In [18]:
# English → Persian mapping
column_name_fa = {
    "city_slug": "شهر",
    "neighborhood_slug": "محله",
    "price_value": "قیمت",
    "building_size": "زیربنا (متر مربع)",
    "rooms_count": "تعداد اتاق",
    "floor": "طبقه",
    "has_balcony": "بالکن",
    "has_elevator": "آسانسور",
    "has_warehouse": "انبار",
    "has_parking": "پارکینگ",
    "has_warm_water_provider": "تامین‌کننده آب گرم",
    "construction_year": "سال ساخت",
    "deed_type": "نوع سند",
    "has_heating_system": "سیستم گرمایشی",
    "has_cooling_system": "سیستم سرمایشی",
    "has_restroom": "نوع سرویس بهداشتی",
    "building_direction": "جهت ساختمان",
    "floor_material": "جنس کف",
    "total_floors_count": "تعداد کل طبقات",
    "unit_per_floor": "تعداد واحد در هر طبقه",
    "is_rebuilt": "بازسازی شده یا نه"
}

def print_out_of_range_fields(validated_json, col_map=column_name_fa):
    out_of_range_keys = [key for key, val in validated_json.items() if val == "❌out of range"]

    if not out_of_range_keys:
        print("✅همه مقادیر معتبر هستن")
        return

    print("🤨:مقدار های زیر نامعتبر بودن یا درست وارد نشدن")
    
    for key in out_of_range_keys:
        persian_name = col_map.get(key.replace("min_", "").replace("max_", ""), key)
        print(f"- {persian_name}")

    print("آیا مایلی دوباره این موارد رو وارد کنی یا با همین اطلاعات جستجو رو شروع کنیم؟")

validated1 = {
    "city_slug": "❌out of range",
    "price_value": 5000000000,
    "rooms_count": "❌out of range",
    "has_balcony": True,
    "max_floor": "❌out of range"
}

print_out_of_range_fields(validated1, column_name_fa)

🤨:مقدار های زیر نامعتبر بودن یا درست وارد نشدن
- شهر
- تعداد اتاق
- طبقه
آیا مایلی دوباره این موارد رو وارد کنی یا با همین اطلاعات جستجو رو شروع کنیم؟


Testing for the inputs that were previsouly out of range

In [301]:
print_out_of_range_fields(oor1)

🤨:مقدار های زیر نامعتبر بودن یا درست وارد نشدن
- قیمت
آیا مایلی دوباره این موارد رو وارد کنی یا با همین اطلاعات جستجو رو شروع کنیم؟


In [302]:
print_out_of_range_fields(sample_range1)

✅همه مقادیر معتبر هستن


# Enabling the user having a conversation with LLM

In [11]:
df = pd.read_csv('filtered_df_cleaned_v2.csv')
df.info()

C:\Users\Parinaz\AppData\Local\Temp\ipykernel_2496\2775316985.py:1: DtypeWarning: Columns (18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('filtered_df_cleaned_v2.csv')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303385 entries, 0 to 303384
Data columns (total 34 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   cat2_slug                303385 non-null  object 
 1   cat3_slug                303385 non-null  object 
 2   city_slug                303385 non-null  object 
 3   neighborhood_slug        167946 non-null  object 
 4   user_type                110664 non-null  object 
 5   description              303385 non-null  object 
 6   title                    303372 non-null  object 
 7   rent_mode                0 non-null       float64
 8   price_mode               303383 non-null  object 
 9   price_value              303380 non-null  float64
 10  credit_mode              0 non-null       float64
 11  building_size            303380 non-null  float64
 12  deed_type                185556 non-null  object 
 13  floor                    303377 non-null  float64
 14  room

In [13]:
column_name_fa

{'city_slug': 'شهر',
 'neighborhood_slug': 'محله',
 'price_value': 'قیمت',
 'building_size': 'زیربنا (متر مربع)',
 'rooms_count': 'تعداد اتاق',
 'floor': 'طبقه',
 'has_balcony': 'بالکن',
 'has_elevator': 'آسانسور',
 'has_warehouse': 'انبار',
 'has_parking': 'پارکینگ',
 'has_warm_water_provider': 'تامین\u200cکننده آب گرم',
 'construction_year': 'سال ساخت',
 'deed_type': 'نوع سند',
 'has_heating_system': 'سیستم گرمایشی',
 'has_cooling_system': 'سیستم سرمایشی',
 'has_restroom': 'نوع سرویس بهداشتی',
 'building_direction': 'جهت ساختمان',
 'floor_material': 'جنس کف',
 'total_floors_count': 'تعداد کل طبقات',
 'unit_per_floor': 'تعداد واحد در هر طبقه',
 'is_rebuilt': 'بازسازی شده یا نه'}

Handling the user entering **at least 4 features**

In [19]:
COLUMNS_TO_USE = [
    # Main
    "city_slug",
    "neighborhood_slug",
    "price_value",
    "building_size",
    "rooms_count",
    
    # Secondary
    "floor",
    "has_balcony",
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_warm_water_provider",
    "construction_year",
    "deed_type",
    "has_heating_system",
    "has_cooling_system",
    "has_restroom",
    "building_direction",
    "floor_material",
    
    # Optional / rarely used
    "total_floors_count",
    "unit_per_floor",
    "is_rebuilt"
]

In [20]:
scenario2 = "من می‌خوام یه خونه بخرم."
user_input = scenario2
extracted = parse_user_query(user_input)
print(extracted)

{'city_slug': None, 'neighborhood_slug': None, 'price_value': None, 'building_size': None, 'rooms_count': None, 'floor': None, 'has_balcony': None, 'has_elevator': None, 'has_warehouse': None, 'has_parking': None, 'has_warm_water_provider': None, 'construction_year': None, 'deed_type': None, 'has_heating_system': None, 'has_cooling_system': None, 'has_restroom': None, 'building_direction': None, 'floor_material': None, 'total_floors_count': None, 'unit_per_floor': None, 'is_rebuilt': None}


Adding to the prompt and trying again:

In [21]:
def parse_user_query(user_text):
    prompt = f"""
        You are a JSON field extractor.
        The input will be a Persian text describing a property search.
        Your task:
        - Read the Persian text carefully.
        - Identify which fields from this dataset columns list the user mentioned: {COLUMNS_TO_USE}
        - Only assign a numeric value to a column if the text clearly states that number in direct connection with that column’s meaning
        - Only include columns that are explicitly mentioned or implied.
        - Extract the exact value for each mentioned column from the sentence.
        - If a value is numeric, return it as a number (not a string).
        - If the user specifies a range like "حداقل", "حداکثر", "کمتر از", "بیشتر از", then instead of the exact column name, use "min_" or "max_" prefix. 
          Examples:
            - "حداقل ۸۰۰ میلیون" → "min_price_value": 800000000
            - "تا ۸۰۰ میلیون" → "max_price_value": 800000000
        
        - Output valid JSON only, with keys exactly as in the list and values.
        - Omit any field not mentioned.
        - If a number is given in Persian words (یک, دو, سه), convert to integer.
        - Ignore irrelevant text.

        ⚠️ Important rules for output:
        - Only output valid JSON.
        - Do NOT include any key with null/None/empty values.
        - If no fields are detected, return an empty JSON: {{}}

        Some of the columns and their allowed values:
        - deed_type: ["single_page", "written_agreement", "multi_page", "other"]
        - has_warm_water_provider: ["water_heater", "package", "powerhouse"]
        - has_heating_system: ["shoofaj", "heater", "duct_split", "floor_heating", "fan_coil", "split", "fireplace"]
        - has_cooling_system: ["air_conditioner", "water_cooler", "duct_split", "split", "fan_coil"]
        - has_restroom: ["squat_seat", "squat", "seat"]
        - floor_material: ["mosaic", "ceramic", "stone", "wood_parquet", "laminate_parquet", "carpet", "floor_covering"    
        
        Example:
        User: "یه خونه سه خوابه تو قلهک تهران می‌خوام، قیمتش تا ۵۰ میلیارد، صد متره، بالکن و آسانسور داره، پارکینگ هم داره، طبقه ۲، سند تک‌برگه، آبگرم‌کن داره، سال ساخت ۱۴۰۰، شوفاژ و کولر داره، دستشویی ایرانیه، به سمت جنوبه، کفش موزاییکه، نوع ملک معمولیه، کل ساختمان ۴ طبقه و هر طبقه ۲ واحد، انبار نداره و بازسازی نشده"
        Output:
        {{
          "city_slug": "tehran",
          "neighborhood_slug": "gholhak",
          "rooms_count": 3,
          "price_value": 50000000000,
          "building_size": 100,
          "has_balcony": True,
          "has_elevator": True,
          "has_parking": True,
          "floor": 2,
          "deed_type": "single_page",
          "has_warm_water_provider": "water_heater",
          "construction_year": 1400,
          "has_heating_system": "shoofaj",
          "has_cooling_system": "air_conditioner",
          "has_restroom": "squat_seat",
          "building_direction": "south",
          "floor_material": "mosaic",
          "total_floors_count": 5,
          "unit_per_floor": 2,
          "is_rebuilt": False,
          "has_warehouse": False,
          "total_floors_count": 4
        }}
        """

    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You output JSON only."},
            {"role": "user", "content": prompt + "\nUser: " + user_text}
        ]
    )
    
    content = completion.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {"Empty"}

In [31]:
# Fixed!

scenario2_again = "من می‌خوام یه خونه بخرم."
user_input = scenario2_again
extracted2 = parse_user_query(user_input)
print(extracted2)

{}


In [23]:
# just testing

sample = " من می‌خوام یه خونه بخرم با تعداد اتاق بین 1 تا 4 تا اتاق داشته باشه."
user_input = sample
extracted = parse_user_query(user_input)
print(extracted) # correct!

{'min_rooms_count': 1, 'max_rooms_count': 4}


Now specifying what to output to user if **the json is empty** (meaning that the user didn't specify things)

In [24]:
print('من دستیار جستجوی خانه هستم، در کدام شهر خانه میخواهی؟')

من دستیار جستجوی خانه هستم، در کدام شهر خانه میخواهی؟


### Making api interactive

It actually needs to be done in the Telegram bot

In [19]:
import json

# Order of asking
FIELDS_ORDER = [
    "city_slug", "neighborhood_slug", "price_value", "building_size",
    "rooms_count", "floor", "has_parking", "has_elevator", "has_balcony",
    "has_warehouse", "construction_year", "deed_type", "has_warm_water_provider",
    "has_heating_system", "has_cooling_system", "has_restroom", "building_direction",
    "floor_material", "total_floors_count", "unit_per_floor", "is_rebuilt"
]

def check_json_fields(json_data, col_map=column_name_fa, min_required=4):
    """Check for out-of-range + missing fields. Print friendly messages."""
    
    # --- Out of range handling ---
    out_of_range_keys = [key for key, val in json_data.items() if val == "❌out of range"]
    if out_of_range_keys:
        print("🤨 بعضی مقادیر درست وارد نشدن یا خارج از محدوده بودن:")
        for key in out_of_range_keys:
            persian_name = col_map.get(key.replace("min_", "").replace("max_", ""), key)
            print(f"- {persian_name}")
        print("👉 می‌خوای دوباره اینا رو وارد کنی یا با همین اطلاعات ادامه بدیم؟\n")
    
    # --- Count how many valid fields are filled ---
    filled_fields = {
        k: v for k, v in json_data.items()
        if v not in (None, "", [], "❌out of range")
    }
    num_filled = len(filled_fields)

    if num_filled >= min_required:
        print(f"✅ حداقل {min_required} فیلد پر شده، می‌تونیم ادامه بدیم.")
        return json_data

    # --- If not enough, suggest missing ones in logical order ---
    print(f"⚠️فقط {num_filled} مورد مشخص شده. لازمه حداقل {min_required} فیلد برای جستجو مشخص باشه")
    print("👇این گزینه‌ ها هنوز مشخص نشدن. هر کدوم رو که میخوای مشخص کن")
    
    missing = [
        (key, col_map.get(key, key))
        for key in FIELDS_ORDER
        if key not in filled_fields
    ]
    
    for key, label in missing:
        print(f"- {label} ({key})")

    return missing

In [49]:
range_test(extracted)

Testing for: {'min_rooms_count': 1, 'max_rooms_count': 4} 
------------------------------------------


{'min_rooms_count': 1, 'max_rooms_count': 4}

In [32]:
sn2 = range_test(extracted2)
sn2

Testing for: {} 
------------------------------------------


{}

In [33]:
print_out_of_range_fields(sn2, column_name_fa)

✅همه مقادیر معتبر هستن


In [47]:
# testing for scenario 2
missing_sn2 = check_json_fields(sn2)

⚠️فقط 0 مورد مشخص شده. لازمه حداقل 4 فیلد برای جستجو مشخص باشه
👇این گزینه‌ ها هنوز مشخص نشدن. کدومشون رو می‌خوای مشخص کنی؟
- شهر (city_slug)
- محله (neighborhood_slug)
- قیمت (price_value)
- زیربنا (متر مربع) (building_size)
- تعداد اتاق (rooms_count)
- طبقه (floor)
- پارکینگ (has_parking)
- آسانسور (has_elevator)
- بالکن (has_balcony)
- انبار (has_warehouse)
- سال ساخت (construction_year)
- نوع سند (deed_type)
- تامین‌کننده آب گرم (has_warm_water_provider)
- سیستم گرمایشی (has_heating_system)
- سیستم سرمایشی (has_cooling_system)
- نوع سرویس بهداشتی (has_restroom)
- جهت ساختمان (building_direction)
- جنس کف (floor_material)
- تعداد کل طبقات (total_floors_count)
- تعداد واحد در هر طبقه (unit_per_floor)
- بازسازی شده یا نه (is_rebuilt)


**Note**: Having these, I can just ask the user to "Please intput ____" in a friendly manner.

# Returning the query results and displaying for the user

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303385 entries, 0 to 303384
Data columns (total 34 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   cat2_slug                303385 non-null  object 
 1   cat3_slug                303385 non-null  object 
 2   city_slug                303385 non-null  object 
 3   neighborhood_slug        167946 non-null  object 
 4   user_type                110664 non-null  object 
 5   description              303385 non-null  object 
 6   title                    303372 non-null  object 
 7   rent_mode                0 non-null       float64
 8   price_mode               303383 non-null  object 
 9   price_value              303380 non-null  float64
 10  credit_mode              0 non-null       float64
 11  building_size            303380 non-null  float64
 12  deed_type                185556 non-null  object 
 13  floor                    303377 non-null  float64
 14  room

In [13]:
def filter_dataframe(df: pd.DataFrame, user_filters: dict):
    filtered_df = df.copy()

    # --- First, group min_/max_ constraints per column ---
    range_constraints = {}
    for key, value in user_filters.items():
        if value in [None, "", "❌out of range"]:
            continue
        if key.startswith("min_") or key.startswith("max_"):
            base_col = key.replace("min_", "").replace("max_", "")
            if base_col not in df.columns:
                continue
            if base_col not in range_constraints:
                range_constraints[base_col] = {}
            if key.startswith("min_"):
                range_constraints[base_col]["min"] = value
            else:
                range_constraints[base_col]["max"] = value

    # --- Apply all filters ---
    for key, value in user_filters.items():
        if value in [None, "", "❌out of range"]:
            continue

        # --- Handle ranges together ---
        if key.startswith("min_") or key.startswith("max_"):
            base_col = key.replace("min_", "").replace("max_", "")
            if base_col not in df.columns:
                continue

            bounds = range_constraints.get(base_col, {})
            if "min" in bounds and "max" in bounds:
                filtered_df = filtered_df[
                    (filtered_df[base_col] >= bounds["min"]) &
                    (filtered_df[base_col] <= bounds["max"])
                ]
            elif "min" in bounds:
                filtered_df = filtered_df[filtered_df[base_col] >= bounds["min"]]
            elif "max" in bounds:
                filtered_df = filtered_df[filtered_df[base_col] <= bounds["max"]]

            # prevent double filtering
            range_constraints.pop(base_col, None)
            continue

        # --- Handle booleans (stored as object strings sometimes) ---
        elif isinstance(value, bool):
            if key in df.columns:
                filtered_df = filtered_df[
                    filtered_df[key].astype(str).str.lower() == str(value).lower()
                ]

        
        # --- Handle string similarity (city, neighborhood, etc.) ---
        elif isinstance(value, str) and key in filtered_df.columns:
            # normalize candidates to lowercase
            candidates = filtered_df[key].dropna().astype(str).str.lower().unique()
            val_lower = value.lower()
            
            # 1. substring match: user input is part of candidate or vice versa
            substring_matches = [c for c in candidates if val_lower in c or c in val_lower]
            
            if substring_matches:
                match_str = min(substring_matches, key=len)  # pick shortest match
            else:
                # 2. fallback to difflib similarity
                match = difflib.get_close_matches(val_lower, candidates, n=1, cutoff=0.7) #  0.7
                match_str = match[0] if match else None
        
            # apply filter if a match was found
            if match_str:
                filtered_df = filtered_df[filtered_df[key].str.lower() == match_str]

        # --- Handle direct matches for other columns ---
        elif key in df.columns:
            filtered_df = filtered_df[filtered_df[key] == value]

    return filtered_df

In [67]:
user_filters = {
    'city_slug': 'mashad', # typo
    'neighborhood_slug': 'hashemieh',
    'min_rooms_count': 2,
    'max_price_value': '❌out of range',
    'has_balcony': True
}

results = filter_dataframe(df, user_filters)

print(f"✅ Found {len(results)} matching listings")
results.head()

✅ Found 304 matching listings


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,user_type,description,title,rent_mode,price_mode,price_value,...,has_heating_system,has_cooling_system,has_restroom,building_direction,floor_material,location_latitude,location_longitude,location_radius,year_created,month_created
352,residential-sell,apartment-sell,mashhad,hashemieh,NaN,**********املاک کومه***********\nآپارتمان۱۲۵مت...,۱۲۵ متری /دوخوابه/ابتدای هاشمیه/فول,NaN,مقطوع,7.300000e+09,...,shoofaj,duct_split,squat_seat,south,ceramic,NaN,NaN,NaN,2024,12
528,residential-sell,apartment-sell,mashhad,hashemieh,NaN,✅آپارتمان ۱۳۴متری صفر تک واحدی هاشمیه سامانیه(...,آپارتمان ۱۳۴متری تک واحدی صفر هاشمیه سامانیه,NaN,مقطوع,9.246000e+09,...,duct_split,duct_split,squat_seat,north,ceramic,NaN,NaN,NaN,2024,11
1258,residential-sell,apartment-sell,mashhad,hashemieh,NaN,## saeed babania ##\n\n #این ...,آپارتمان۴۵۰متری/هاشمیه۱۴/Brand,NaN,مقطوع,6.750000e+10,...,duct_split,duct_split,squat_seat,north,stone,36.317257,59.510944,NaN,2024,12
3036,residential-sell,apartment-sell,mashhad,hashemieh,NaN,بنام خدا\n\nبسیار خوش نقشه و نورگیر\n\nلابی بی...,۱۵۵متر اول هاشمیه اسانسور لابی فول خوش نقشه,NaN,مقطوع,8.500000e+09,...,shoofaj,air_conditioner,squat_seat,south,stone,NaN,NaN,NaN,2024,12
5030,residential-sell,apartment-sell,mashhad,hashemieh,NaN,۸۵ متر دو خوابه - خوش نقشه - جنوبی - نما سنگ \...,۸۵ متر دوخوابه - هاشمیه,NaN,مقطوع,2.250000e+09,...,shoofaj,air_conditioner,squat,south,ceramic,36.295792,59.491222,NaN,2024,7


In [73]:
user_filters = {
    'city_slug': 'mashad', # typo
    'neighborhood_slug': 'hashemieh',
    'min_rooms_count': 2,
    'max_price_value': '❌out of range',
    'has_balcony': True
}

results = filter_dataframe(df, user_filters)

print(f"✅ Found {len(results)} matching listings")
results.head()

✅ Found 304 matching listings


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,user_type,description,title,rent_mode,price_mode,price_value,...,has_heating_system,has_cooling_system,has_restroom,building_direction,floor_material,location_latitude,location_longitude,location_radius,year_created,month_created
352,residential-sell,apartment-sell,mashhad,hashemieh,NaN,**********املاک کومه***********\nآپارتمان۱۲۵مت...,۱۲۵ متری /دوخوابه/ابتدای هاشمیه/فول,NaN,مقطوع,7.300000e+09,...,shoofaj,duct_split,squat_seat,south,ceramic,NaN,NaN,NaN,2024,12
528,residential-sell,apartment-sell,mashhad,hashemieh,NaN,✅آپارتمان ۱۳۴متری صفر تک واحدی هاشمیه سامانیه(...,آپارتمان ۱۳۴متری تک واحدی صفر هاشمیه سامانیه,NaN,مقطوع,9.246000e+09,...,duct_split,duct_split,squat_seat,north,ceramic,NaN,NaN,NaN,2024,11
1258,residential-sell,apartment-sell,mashhad,hashemieh,NaN,## saeed babania ##\n\n #این ...,آپارتمان۴۵۰متری/هاشمیه۱۴/Brand,NaN,مقطوع,6.750000e+10,...,duct_split,duct_split,squat_seat,north,stone,36.317257,59.510944,NaN,2024,12
3036,residential-sell,apartment-sell,mashhad,hashemieh,NaN,بنام خدا\n\nبسیار خوش نقشه و نورگیر\n\nلابی بی...,۱۵۵متر اول هاشمیه اسانسور لابی فول خوش نقشه,NaN,مقطوع,8.500000e+09,...,shoofaj,air_conditioner,squat_seat,south,stone,NaN,NaN,NaN,2024,12
5030,residential-sell,apartment-sell,mashhad,hashemieh,NaN,۸۵ متر دو خوابه - خوش نقشه - جنوبی - نما سنگ \...,۸۵ متر دوخوابه - هاشمیه,NaN,مقطوع,2.250000e+09,...,shoofaj,air_conditioner,squat,south,ceramic,36.295792,59.491222,NaN,2024,7


In [76]:
results['rooms_count'].value_counts()  # correct! it's more than 2

rooms_count
3.0    167
2.0    118
4.0     17
6.0      2
Name: count, dtype: int64

### Testing on user input process

In [72]:
df[df['city_slug'] == 'mashhad'].neighborhood_slug.value_counts()

neighborhood_slug
elahiyehblvd        3924
ghasemabad          2286
bolvartoos          1249
azadshahr           1045
north-tabars         794
                    ... 
kuy-e-police           2
dehnavi                2
Shahrak-e-Shirin       2
mooud                  1
neyzeh                 1
Name: count, Length: 147, dtype: int64

**Note**: It needs work for `elahiyehblvd` and `elahiyeh` being the same 

In [88]:
test1 = "یک خونه در شهر مشهد قاسم آباد میخوام که بین یک تا 4 تا اتاق داشته باشه و آسانسور هم داشته باشه، شوفاژ هم خوبه واقعا. با بالکن ولی هیچ پارکینگی نداشته باشه"

user_input = test1
extracted_t1 = parse_user_query(user_input)
print("---API Extracted---")
print(extracted_t1)

# --- range check ---
t1_filtered = range_test(extracted_t1)
print("\n---Range Test Result---")
print(t1_filtered)

results = filter_dataframe(df, t1_filtered)

print(f"✅ Found {len(results)} matching listings")
results.head()

---API Extracted---
{'city_slug': 'mashhad', 'neighborhood_slug': 'qasemabad', 'min_rooms_count': 1, 'max_rooms_count': 4, 'has_elevator': True, 'has_balcony': True, 'has_parking': False, 'has_heating_system': 'shoofaj'}
Testing for: {'city_slug': 'mashhad', 'neighborhood_slug': 'qasemabad', 'min_rooms_count': 1, 'max_rooms_count': 4, 'has_elevator': True, 'has_balcony': True, 'has_parking': False, 'has_heating_system': 'shoofaj'} 
------------------------------------------

---Range Test Result---
{'city_slug': 'mashhad', 'neighborhood_slug': 'qasemabad', 'min_rooms_count': 1, 'max_rooms_count': 4, 'has_elevator': True, 'has_balcony': True, 'has_parking': False, 'has_heating_system': 'shoofaj'}
✅ Found 11 matching listings


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,user_type,description,title,rent_mode,price_mode,price_value,...,has_heating_system,has_cooling_system,has_restroom,building_direction,floor_material,location_latitude,location_longitude,location_radius,year_created,month_created
52922,residential-sell,apartment-sell,mashhad,ghasemabad,NaN,موقعیت مکانی و دسترسی به امکانات عالی\nنگهبانی...,آپارتمان ۷۸ متری مجتمع آفتاب بلوار اندیشه,NaN,مقطوع,2.200000e+09,...,shoofaj,water_cooler,squat,NaN,floor_covering,36.377838,59.521755,0.0,2024,11
57354,residential-sell,apartment-sell,mashhad,ghasemabad,NaN,باسلام آپارتمان ۷۵ متر \nطبقه دوم با اسانسور \...,اپارتمان ۷۵ متری اندیشه,NaN,مقطوع,1.800000e+09,...,shoofaj,air_conditioner,squat,south,ceramic,36.381550,59.518112,NaN,2024,9
77462,residential-sell,apartment-sell,mashhad,ghasemabad,مشاور املاک,بنام خدا\nیک واحد اپارتمان هشتاد و پنج متری بس...,اپارتمان هشتاد و پنج متری فول اندیشه,NaN,مقطوع,2.050000e+09,...,shoofaj,air_conditioner,squat_seat,north,ceramic,36.369167,59.525791,500.0,2024,7
101313,residential-sell,apartment-sell,mashhad,ghasemabad,NaN,میدان نمایشگاه خیابان نهضت حاشیهءخیابان۲۰متری ...,آپارتمان۷۰متردوخوابه,NaN,مقطوع,1.680000e+09,...,shoofaj,NaN,squat_seat,south,ceramic,NaN,NaN,NaN,2024,10
104182,residential-sell,apartment-sell,mashhad,ghasemabad,NaN,مهندسی ساز قولنامه با قیمت مناسب واقع در بهتر...,آپارتمان ۶۵ متری دو خواب شیک و به روز,NaN,مقطوع,1.480000e+09,...,shoofaj,air_conditioner,squat_seat,south,ceramic,36.348999,59.473869,NaN,2024,11


In [89]:
results[results['neighborhood_slug'] != 'ghasemabad'] # correct! all of the neighborhoods

,cat2_slug,cat3_slug,city_slug,neighborhood_slug,user_type,description,title,rent_mode,price_mode,price_value,...,has_heating_system,has_cooling_system,has_restroom,building_direction,floor_material,location_latitude,location_longitude,location_radius,year_created,month_created


In [92]:
results[['has_balcony', 'has_parking', 'has_elevator']].value_counts() # All correct!

has_balcony  has_parking  has_elevator
True         False        True            11
Name: count, dtype: int64

## Handling actual interation

In [3]:
df = pd.read_csv('filtered_df_cleaned_v2.csv')
df.head()

C:\Users\Parinaz\AppData\Local\Temp\ipykernel_15000\97826982.py:1: DtypeWarning: Columns (18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('filtered_df_cleaned_v2.csv') # DtypeWarning: Columns (10,26,28,52) have mixed types.


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,user_type,description,title,rent_mode,price_mode,price_value,...,has_heating_system,has_cooling_system,has_restroom,building_direction,floor_material,location_latitude,location_longitude,location_radius,year_created,month_created
0,residential-sell,apartment-sell,tehran,gholhak,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,مقطوع,8.500000e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0,2024,5
1,residential-sell,apartment-sell,mashhad,emamreza,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,NaN,مقطوع,5.750000e+09,...,shoofaj,air_conditioner,squat_seat,north,ceramic,NaN,NaN,NaN,2024,5
2,residential-sell,apartment-sell,tehran,dardasht,مشاور املاک,♡♡♡♡♡♡بنام خدا♡♡♡♡♡♡\n♡♡♡عرض ادب واحترام♡♡♡\n♡...,اپارتمان ۱۰۰متری طبقه۳و۴ فول دردشت شمالی,NaN,مقطوع,8.700000e+09,...,NaN,NaN,NaN,NaN,NaN,35.729832,51.505466,500.0,2024,9
3,residential-sell,apartment-sell,mahdasht-city,NaN,مشاور املاک,با سلام\n\nاملاک بزرگ اتحاد با ۲شعبه فعال با ف...,۷۸متری دوبحرنورگیر بازسازی شده شیک سندتک برگ,NaN,مقطوع,6.500000e+08,...,shoofaj,water_cooler,squat,north,ceramic,35.712364,50.794781,500.0,2024,6
4,residential-sell,apartment-sell,mashhad,faramarzabbasi,NaN,اول رسالت فرد نزدیک به حاشیه فرامرز .. ساکنین...,متراژ ۸۰ متر فرامرز نزدیک حاشیه,NaN,مقطوع,3.000000e+09,...,NaN,NaN,squat,NaN,ceramic,NaN,NaN,NaN,2024,10


In [6]:
COLUMNS_TO_USE = [
    # Main
    "city_slug",
    "neighborhood_slug",
    "price_value",
    "building_size",
    "rooms_count",
    
    # Secondary
    "floor",
    "has_balcony",
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_warm_water_provider",
    "construction_year",
    "deed_type",
    "has_heating_system",
    "has_cooling_system",
    "has_restroom",
    "building_direction",
    "floor_material",
    
    # Optional / rarely used
    "total_floors_count",
    "unit_per_floor",
    "is_rebuilt"
]

In [7]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
)

In [32]:
def parse_user_query(user_text):
    prompt = f"""
        You are a JSON field extractor.
        The input will be a Persian text describing a property search.
        Your task:
        - Read the Persian text carefully.
        - Identify which fields from this dataset columns list the user mentioned: {COLUMNS_TO_USE}
        - Only assign a numeric value to a column if the text clearly states that number in direct connection with that column’s meaning
        - Only include columns that are explicitly mentioned or implied.
        - Extract the exact value for each mentioned column from the sentence.
        - If a value is numeric, return it as a number (not a string).
        - If the user specifies a range like "حداقل", "حداکثر", "کمتر از", "بیشتر از", then instead of the exact column name, use "min_" or "max_" prefix. 
          Examples:
            - "حداقل ۸۰۰ میلیون" → "min_price_value": 800000000
            - "تا ۸۰۰ میلیون" → "max_price_value": 800000000
        
        - Output valid JSON only, with keys exactly as in the list and values.
        - Omit any field not mentioned.
        - If a number is given in Persian words (یک, دو, سه), convert to integer.
        - Ignore irrelevant text.

        ⚠️ Important rules for output:
        - Only output valid JSON.
        - Do NOT include any key with null/None/empty values.
        - If no fields are detected, return an empty JSON: {{}}

        Some of the columns and their allowed values:
        - deed_type: ["single_page", "written_agreement", "multi_page", "other"]
        - has_warm_water_provider: ["water_heater", "package", "powerhouse"]
        - has_heating_system: ["shoofaj", "heater", "duct_split", "floor_heating", "fan_coil", "split", "fireplace"]
        - has_cooling_system: ["air_conditioner", "water_cooler", "duct_split", "split", "fan_coil"]
        - has_restroom: ["squat_seat", "squat", "seat"]
        - floor_material: ["mosaic", "ceramic", "stone", "wood_parquet", "laminate_parquet", "carpet", "floor_covering"
        
        Example:
        User: "یه خونه سه خوابه تو قلهک تهران می‌خوام، قیمتش تا ۵۰ میلیارد، صد متره، بالکن و آسانسور داره، پارکینگ هم داره، طبقه ۲، سند تک‌برگه، آبگرم‌کن داره، سال ساخت ۱۴۰۰، شوفاژ و کولر داره، دستشویی ایرانیه، به سمت جنوبه، کفش موزاییکه، نوع ملک معمولیه، کل ساختمان ۴ طبقه و هر طبقه ۲ واحد، انبار نداره و بازسازی نشده"
        Output:
        {{
          "city_slug": "tehran",
          "neighborhood_slug": "gholhak",
          "rooms_count": 3,
          "price_value": 50000000000,
          "building_size": 100,
          "has_balcony": True,
          "has_elevator": True,
          "has_parking": True,
          "floor": 2,
          "deed_type": "single_page",
          "has_warm_water_provider": "water_heater",
          "construction_year": 1400,
          "has_heating_system": "shoofaj",
          "has_cooling_system": "air_conditioner",
          "has_restroom": "squat_seat",
          "building_direction": "south",
          "floor_material": "mosaic",
          "total_floors_count": 5,
          "unit_per_floor": 2,
          "is_rebuilt": False,
          "has_warehouse": False,
          "total_floors_count": 4
        }}
        """

    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You output JSON only."},
            {"role": "user", "content": prompt + "\nUser: " + user_text}
        ]
    )
    
    content = completion.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {"Empty"}

In [9]:
sn1 = "سلام. من دنبال یک خونه ویلایی توی هاشمیه مشهد برای اجاره هستم. حداقل دو خواب داشته باشه و پول پیشش بیشتر از ۸۰۰ میلیون نباشه."
user_input = sn1
extracted = parse_user_query(user_input)
print(extracted)

{'city_slug': 'mashhad', 'neighborhood_slug': 'hashemieh', 'min_rooms_count': 2, 'max_price_value': 800000000}


In [11]:
sn1_filtered = range_test(extracted)
sn1_filtered

Testing for: {'city_slug': 'mashhad', 'neighborhood_slug': 'hashemieh', 'min_rooms_count': 2, 'max_price_value': 800000000} 
------------------------------------------


{'city_slug': 'mashhad',
 'neighborhood_slug': 'hashemieh',
 'min_rooms_count': 2,
 'max_price_value': 800000000}

In [14]:
results = filter_dataframe(df, sn1_filtered)

print(f"✅ Found {len(results)} matching listings")
results.head()

✅ Found 12 matching listings


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,user_type,description,title,rent_mode,price_mode,price_value,...,has_heating_system,has_cooling_system,has_restroom,building_direction,floor_material,location_latitude,location_longitude,location_radius,year_created,month_created
7881,residential-sell,apartment-sell,mashhad,hashemieh,NaN,.............................پیام نده زنگ بزن....,اجاره وفروشی/۳خوابه/هاشمیه/زیرقیمت,NaN,مقطوع,1111111.0,...,heater,water_cooler,squat_seat,north,ceramic,NaN,NaN,NaN,2024,12
14734,residential-sell,apartment-sell,mashhad,hashemieh,مشاور املاک,زیر فی منطقه\n\nفروش به مدت محدود\n\nآپارتمان ...,آپارتمان ۱۳۵ متری صفر سند آماده/ پرواز,NaN,مقطوع,111111111.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,7
33037,residential-sell,apartment-sell,mashhad,hashemieh,NaN,همشریان عزیز\n\n اگر قصد خرید و فروش و ...,آپارتمان ویلایی معاوضه‌ی خود را به ما بسپارید,NaN,مقطوع,111111.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,12
38825,residential-sell,apartment-sell,mashhad,hashemieh,NaN,سند تک برگ.\nخانه تخلیه.\nدوستان املاکی لطفا ف...,هاشمیه اپارتمان,NaN,مقطوع,3400000.0,...,heater,water_cooler,squat,NaN,ceramic,NaN,NaN,NaN,2024,10
51660,residential-sell,apartment-sell,mashhad,hashemieh,NaN,آپارتمان ۲خ فوق العاده شیک در هاشمیه نزدیک باز...,آپارتمان هاشمیه ۸۵ ۲خ,NaN,مقطوع,11111111.0,...,shoofaj,water_cooler,squat_seat,north,wood_parquet,NaN,NaN,NaN,2024,11


### Start changing

Making the model **answer** to the user

**Added answer** to the prompt

In [61]:
def parse_user_query(user_text):
    prompt = f"""
        You are a JSON field extractor.
        The input will be a Persian text describing a property search.
        Your task:
        - Read the Persian text carefully.
        - Identify which fields from this dataset columns list the user mentioned: {COLUMNS_TO_USE}
        - Only assign a numeric value to a column if the text clearly states that number in direct connection with that column’s meaning
        - Only include columns that are explicitly mentioned or implied.
        - Extract the exact value for each mentioned column from the sentence.
        - If a value is numeric, return it as a number (not a string).
        - If the user specifies a range like "حداقل", "حداکثر", "کمتر از", "بیشتر از", then instead of the exact column name, use "min_" or "max_" prefix. 
          Examples:
            - "حداقل ۸۰۰ میلیون" → "min_price_value": 800000000
            - "تا ۸۰۰ میلیون" → "max_price_value": 800000000
        
        - Always extract **all valid house-related fields** mentioned in the input, even if there is only one field.
        - Do NOT ignore a valid field just because other fields are missing or irrelevant content is present.
        Example:
            - "سلام، می‌خوام یه خونه در مشهد بخرم" → "city_slug": "mashhad"
        - Output valid JSON only, with keys exactly as in the list and values.
        - Omit any field not mentioned.
        - If a number is given in Persian words (یک, دو, سه), convert to integer.
        - Ignore irrelevant text.

        ⚠️ Important rules for output:
        - Only output valid JSON.
        - Do NOT include any key with null/None/empty values.
        - If no fields are detected, return an empty JSON: {{}}

        Some of the columns and their allowed values:
        - deed_type: ["single_page", "written_agreement", "multi_page", "other"]
        - has_warm_water_provider: ["water_heater", "package", "powerhouse"]
        - has_heating_system: ["shoofaj", "heater", "duct_split", "floor_heating", "fan_coil", "split", "fireplace"]
        - has_cooling_system: ["air_conditioner", "water_cooler", "duct_split", "split", "fan_coil"]
        - has_restroom: ["squat_seat", "squat", "seat"]
        - floor_material: ["mosaic", "ceramic", "stone", "wood_parquet", "laminate_parquet", "carpet", "floor_covering"

        Additionally, always include two special fields in the JSON:
        - "info": one of ["irrelevant", "partially_relevant", "relevant"]
        - "answer": a friendly and natural Persian response string
    
        Rules for "info":
        - "irrelevant": if no valid house-related fields are found
        - "partially_relevant": if the text mixes irrelevant content with valid fields
        - "relevant": if the text only includes valid house-related info
        
        Rules for "answer":
        - If "irrelevant": kindly introduce yourself in Persian. Use **one of several friendly variants**, and pick one naturally related to the input of the user:
          Example: "من دستیار جستجوی خانه هستم 🌸 لطفا اطلاعاتی مثل شهر، محله، قیمت یا متراژ خونه مورد نظرت رو بگو."
        - If "partially_relevant": politely ignore irrelevant parts, **confirm any valid fields you could extract**, and ask for clarification if more important fields are missing.
        - Example:
                - "سلام، می‌خوام یه خونه دو خوابه با سه تا پراید بخرم." → "rooms_count":2 and "answer": "منظور شما را از سه تا پراید متوجه نشدم. لطفا اطلاعاتی مانند شهر، محله، قیمت یا متراژ خونه رو بگو"
        - If "relevant":
          - If fewer than 4 fields extracted → politely confirm, then ask the user to also provide 5 more important fields from this list (in order): {FIELDS_ORDER}.
          - If 4 or more fields extracted → politely confirm and say: 
            "🔎 اطلاعات کافی دریافت شد، در حال جستجو برای نتیجه‌ها..."
        
        ⚠️ The "answer" must always be included as a JSON string value.  
        Do not output anything outside the JSON.
        
        Example:
        User: "یه خونه سه خوابه تو قلهک تهران می‌خوام، قیمتش تا ۵۰ میلیارد، صد متره، بالکن و آسانسور داره، پارکینگ هم داره، طبقه ۲، سند تک‌برگه، آبگرم‌کن داره، سال ساخت ۱۴۰۰، شوفاژ و کولر داره، دستشویی ایرانیه، به سمت جنوبه، کفش موزاییکه، نوع ملک معمولیه، کل ساختمان ۴ طبقه و هر طبقه ۲ واحد، انبار نداره و بازسازی نشده"
        Output:
        {{
          "city_slug": "tehran",
          "neighborhood_slug": "gholhak",
          "rooms_count": 3,
          "price_value": 50000000000,
          "building_size": 100,
          "has_balcony": True,
          "has_elevator": True,
          "has_parking": True,
          "floor": 2,
          "deed_type": "single_page",
          "has_warm_water_provider": "water_heater",
          "construction_year": 1400,
          "has_heating_system": "shoofaj",
          "has_cooling_system": "air_conditioner",
          "has_restroom": "squat_seat",
          "building_direction": "south",
          "floor_material": "mosaic",
          "total_floors_count": 5,
          "unit_per_floor": 2,
          "is_rebuilt": False,
          "has_warehouse": False,
          "total_floors_count": 4
        }}
        """

    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You output JSON only."},
            {"role": "user", "content": prompt + "\nUser: " + user_text}
        ]
    )
    
    content = completion.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {"Empty"}

In [ ]:
# Order of asking
FIELDS_ORDER = [
    "city_slug", "neighborhood_slug", "price_value", "building_size",
    "rooms_count", "floor", "has_parking", "has_elevator", "has_balcony",
    "has_warehouse", "construction_year", "deed_type", "has_warm_water_provider",
    "has_heating_system", "has_cooling_system", "has_restroom", "building_direction",
    "floor_material", "total_floors_count", "unit_per_floor", "is_rebuilt"
]


# English → Persian mapping
column_name_fa = {
    "city_slug": "شهر",
    "neighborhood_slug": "محله",
    "price_value": "قیمت",
    "building_size": "زیربنا (متر مربع)",
    "rooms_count": "تعداد اتاق",
    "floor": "طبقه",
    "has_balcony": "بالکن",
    "has_elevator": "آسانسور",
    "has_warehouse": "انبار",
    "has_parking": "پارکینگ",
    "has_warm_water_provider": "تامین‌کننده آب گرم",
    "construction_year": "سال ساخت",
    "deed_type": "نوع سند",
    "has_heating_system": "سیستم گرمایشی",
    "has_cooling_system": "سیستم سرمایشی",
    "has_restroom": "نوع سرویس بهداشتی",
    "building_direction": "جهت ساختمان",
    "floor_material": "جنس کف",
    "total_floors_count": "تعداد کل طبقات",
    "unit_per_floor": "تعداد واحد در هر طبقه",
    "is_rebuilt": "بازسازی شده یا نه"
}

In [30]:
sn1_again = "سلام. من دنبال یک خونه ویلایی توی هاشمیه مشهد برای اجاره هستم. حداقل دو خواب داشته باشه و پول پیشش بیشتر از ۸۰۰ میلیون نباشه."
user_input = sn1_again
extracted = parse_user_query(user_input)
print(extracted)

{'city_slug': 'mashhad', 'neighborhood_slug': 'hashmieh', 'min_rooms_count': 2, 'max_price_value': 800000000, 'info': 'relevant', 'answer': '🔎 اطلاعات کافی دریافت شد، در حال جستجو برای نتیجه\u200cها...'}


In [22]:
print(extracted['answer'])

🔎 اطلاعات کافی دریافت شد، در حال جستجو برای نتیجه‌ها...


In [47]:
sn5 = "سلام، می‌خوام یه خونه در مشهد بخرم."
user_input = sn5
extracted = parse_user_query(user_input)
print(extracted)

{'city_slug': 'mashhad', 'info': 'relevant', 'answer': 'سلام! 😊 لطفا اطلاعاتی مثل محله، قیمت یا متراژ خونه مورد نظرت رو بگو.'}


In [48]:
sn5_again = "سلام، می‌خوام یه خونه دو خوابه بخرم."
user_input = sn5_again
extracted = parse_user_query(user_input)
print(extracted) # Fixed!

{'rooms_count': 2, 'info': 'relevant', 'answer': 'لطفا اطلاعات بیشتری مثل شهر، محله، قیمت یا متراژ خونه مورد نظرت رو بگو.'}


In [53]:
sn5_orig = "سلام، می‌خوام یه خونه دو خوابه با سه تا پراید بخرم."
user_input = sn5_orig
extracted = parse_user_query(user_input)
print(extracted) # fixed

{'rooms_count': 2, 'info': 'partially_relevant', 'answer': 'منظور شما را از سه تا پراید متوجه نشدم. لطفا اطلاعاتی مانند شهر، محله، قیمت یا متراژ خونه رو بگو'}


In [54]:
sn5_orig_test = "سلام، می‌خوام یه خونه توی کرج با یک یخچال و فریزر خوب بخرم."
user_input = sn5_orig_test
extracted = parse_user_query(user_input)
print(extracted) # not fixed

{'info': 'irrelevant', 'answer': 'من دستیار جستجوی خانه هستم 🌸 لطفا اطلاعاتی مثل شهر، محله، قیمت یا متراژ خونه مورد نظرت رو بگو.'}


In [55]:
sn5_orig_test = "سلام، می‌خوام یه خونه با 5 تا اتاق و یک پارکینگ با یک یخچال و فریزر خوب بخرم."
user_input = sn5_orig_test
extracted = parse_user_query(user_input)
print(extracted) # fixed

{'rooms_count': 5, 'has_parking': True, 'info': 'partially_relevant', 'answer': 'منظور شما را از یخچال و فریزر متوجه نشدم. لطفا اطلاعاتی مانند شهر، محله، قیمت یا متراژ خونه رو بگو.'}


In [57]:
sn5_orig_test = "سلام، می‌خوام یه خونه 3 خوابه توی تهران با یک یخچال و فریزر خوب بخرم."
user_input = sn5_orig_test
extracted = parse_user_query(user_input)
print(extracted) # not fixed

{'info': 'partially_relevant', 'rooms_count': 3, 'answer': 'منظور شما را از یخچال و فریزر متوجه نشدم. لطفا اطلاعاتی مانند شهر، محله، قیمت یا متراژ خونه رو بگو'}


In [58]:
sn5_orig_test = "سلام، می‌خوام یه خونه 3 خوابه توی تهران بخرم."
user_input = sn5_orig_test
extracted = parse_user_query(user_input)
print(extracted) # fixed

{'city_slug': 'tehran', 'rooms_count': 3, 'info': 'partially_relevant', 'answer': 'لطفا اطلاعاتی مانند قیمت، متراژ، محله، پارکینگ و آسانسور رو هم بفرمایید.'}


## Correcting the interaction

**Removing from prompt** what is unnecessary (which is `info` and `answer`)

In [106]:
def parse_user_query(user_text):
    prompt = f"""
        You are a JSON field extractor.
        The input will be a Persian text describing a property search.
        Your task:
        - Read the Persian text carefully.
        - Identify which fields from this dataset columns list the user mentioned: {COLUMNS_TO_USE}
        - Only assign a numeric value to a column if the text clearly states that number in direct connection with that column’s meaning
        - Only include columns that are explicitly mentioned or implied.
        - Extract the exact value for each mentioned column from the sentence.
        - If a value is numeric, return it as a number (not a string).
        - If the user specifies a range like "حداقل", "حداکثر", "کمتر از", "بیشتر از", then instead of the exact column name, use "min_" or "max_" prefix. 
          Examples:
            - "حداقل ۸۰۰ میلیون" → "min_price_value": 800000000
            - "تا ۸۰۰ میلیون" → "max_price_value": 800000000

        - Output valid JSON only, with keys exactly as in the list and values.
        - Omit any field not mentioned.
        - If a number is given in Persian words (یک, دو, سه), convert to integer.
        - Ignore irrelevant text.

        ⚠️ Important rules for output:
        - Always extract **all valid house-related fields** mentioned in the input, even if there is only one field.
        - Do NOT ignore a valid field just because other fields are missing or irrelevant content is present.
          Example:
            - "سلام، می‌خوام یه خونه در مشهد بخرم" → "city_slug": "mashhad"
            - "یه خونه در تهران می‌خوام" → "city_slug": "tehran"
            - "خونه‌ای توی اصفهان برای اجاره می‌خوام" → "city_slug": "isfahan"
            - "می‌خوام یه آپارتمان در شیراز بخرم" → "city_slug": "shiraz"
        - Only output valid JSON.
        - Do NOT include any key with null/None/empty values.
        - If no fields are detected, return an empty JSON: {{}}

        Some of the columns and their allowed values:
        - deed_type: ["single_page", "written_agreement", "multi_page", "other"]
        - has_warm_water_provider: ["water_heater", "package", "powerhouse"]
        - has_heating_system: ["shoofaj", "heater", "duct_split", "floor_heating", "fan_coil", "split", "fireplace"]
        - has_cooling_system: ["air_conditioner", "water_cooler", "duct_split", "split", "fan_coil"]
        - has_restroom: ["squat_seat", "squat", "seat"]
        - floor_material: ["mosaic", "ceramic", "stone", "wood_parquet", "laminate_parquet", "carpet", "floor_covering"
        
        Example:
        User: "یه خونه سه خوابه تو قلهک تهران می‌خوام، قیمتش تا ۵۰ میلیارد، صد متره، بالکن و آسانسور داره، پارکینگ هم داره، طبقه ۲، سند تک‌برگه، آبگرم‌کن داره، سال ساخت ۱۴۰۰، شوفاژ و کولر داره، دستشویی ایرانیه، به سمت جنوبه، کفش موزاییکه، نوع ملک معمولیه، کل ساختمان ۴ طبقه و هر طبقه ۲ واحد، انبار نداره و بازسازی نشده"
        Output:
        {{
          "city_slug": "tehran",
          "neighborhood_slug": "gholhak",
          "rooms_count": 3,
          "price_value": 50000000000,
          "building_size": 100,
          "has_balcony": True,
          "has_elevator": True,
          "has_parking": True,
          "floor": 2,
          "deed_type": "single_page",
          "has_warm_water_provider": "water_heater",
          "construction_year": 1400,
          "has_heating_system": "shoofaj",
          "has_cooling_system": "air_conditioner",
          "has_restroom": "squat_seat",
          "building_direction": "south",
          "floor_material": "mosaic",
          "total_floors_count": 5,
          "unit_per_floor": 2,
          "is_rebuilt": False,
          "has_warehouse": False,
          "total_floors_count": 4
        }}
        """

    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You output JSON only."},
            {"role": "user", "content": prompt + "\nUser: " + user_text}
        ]
    )
    
    content = completion.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {}

In [79]:
# testing the new prompt

sn5_again = "سلام دنبال یک خونه ای هستم که تو کرج باشه"
user_input = sn5_again
extracted = parse_user_query(user_input)
print(extracted) # Fixed!

{'city_slug': 'karaj'}


In [81]:
sn5_again = "سلام یک خونه ای میخوام که بین 2 تا 6 تا اتاق داشته باشه"
user_input = sn5_again
extracted = parse_user_query(user_input)
print(extracted) # Fixed!

{'min_rooms_count': 2, 'max_rooms_count': 6}


### Checking to see how many is filled

In [96]:
def check_json_fields(json_data, col_map=column_name_fa, min_required=4):
    """Check for out-of-range + missing fields. Print friendly messages."""
    
    # --- Out of range handling ---
    invalid = []
    out_of_range_keys = [key for key, val in json_data.items() if val == "❌out of range"]
    if out_of_range_keys:
        print("invalid data found")
        for key in out_of_range_keys:
            persian_name = col_map.get(key.replace("min_", "").replace("max_", ""), key)
            print(f"- {persian_name}")
            invalid.append(key)

    # --- Normalize min_/max_ fields so they count as the base field ---
    normalized_fields = set()
    for key, val in json_data.items():
        if val not in (None, "", [], "❌out of range"):
            base_key = key.replace("min_", "").replace("max_", "")
            normalized_fields.add(base_key)

    # --- Count how many valid fields are filled ---
    num_filled = len(normalized_fields)

    if num_filled >= min_required:
        print(f"✅ حداقل {min_required} فیلد پر شده، می‌تونیم ادامه بدیم.")
        return json_data

    # --- If not enough, suggest missing ones in logical order ---
    print(f"⚠️ فقط {num_filled} مورد مشخص شده. لازمه حداقل {min_required} فیلد برای جستجو مشخص باشه")
    print("👇 این گزینه‌ ها هنوز مشخص نشدن. هر کدوم رو که میخوای مشخص کن")

    missing = [
        (key, col_map.get(key, key))
        for key in FIELDS_ORDER
        if key not in normalized_fields
    ]
    
    for key, label in missing:
        print(f"- {label} ({key})")

    return invalid, missing, num_filled

In [94]:
validated_range = range_test(extracted)

Testing for: {'min_rooms_count': 2, 'max_rooms_count': 6} 
------------------------------------------


In [95]:
out_of_range, missing, num_filled = check_json_fields(validated_range)

print("--- out of range ---\n", out_of_range)
print("--- missing ---\n", missing)
print("--- number filled ---\n", num_filled)

⚠️ فقط 1 مورد مشخص شده. لازمه حداقل 4 فیلد برای جستجو مشخص باشه
👇 این گزینه‌ ها هنوز مشخص نشدن. هر کدوم رو که میخوای مشخص کن
- شهر (city_slug)
- محله (neighborhood_slug)
- قیمت (price_value)
- زیربنا (متر مربع) (building_size)
- طبقه (floor)
- پارکینگ (has_parking)
- آسانسور (has_elevator)
- بالکن (has_balcony)
- انبار (has_warehouse)
- سال ساخت (construction_year)
- نوع سند (deed_type)
- تامین‌کننده آب گرم (has_warm_water_provider)
- سیستم گرمایشی (has_heating_system)
- سیستم سرمایشی (has_cooling_system)
- نوع سرویس بهداشتی (has_restroom)
- جهت ساختمان (building_direction)
- جنس کف (floor_material)
- تعداد کل طبقات (total_floors_count)
- تعداد واحد در هر طبقه (unit_per_floor)
- بازسازی شده یا نه (is_rebuilt)
--- out of range ---
 []
--- missing ---
 [('city_slug', 'شهر'), ('neighborhood_slug', 'محله'), ('price_value', 'قیمت'), ('building_size', 'زیربنا (متر مربع)'), ('floor', 'طبقه'), ('has_parking', 'پارکینگ'), ('has_elevator', 'آسانسور'), ('has_balcony', 'بالکن'), ('has_warehouse', 

## Processing input in loop

In [100]:
# pseudocode


def session():
    request_count = 0
    user_text = 'یک خونه در مشهد با 10 تا اتاق میخوام که آسانسور داشته باشه'
    done = False
    while done != True: # while done is not true
         _, _, done = process(user_text)
         if done:
             break


def process():
    done = False
    extracted = parse_user_query(user_text)
    print('Extracted:', extracted) 
    
    validated_range = range_test(extracted)
    
    out_of_range, missing, num_filled = check_json_fields(validated_range)
    print("--- out of range ---\n", out_of_range)
    print("--- missing ---\n", missing)
    print("--- number filled ---\n", num_filled)
    
    min_required = 4
    if num_filled >= min_required:
        print("filtering...")
        df_results = filter_dataframe(df, validated_range)
        answer = 'پیدا کردن خانه با مشخصات گفته شده'
        # generating success and searching answer from api
        print('api generated')
        done = True
        return df_results, answer, done
    
    # else
    request_count += 1
    if request_count > 3:
        to_print = """
        - تعداد اتاق
        - محله (neighborhood_slug)
        - قیمت (price_value)
        - زیربنا (متر مربع) (building_size)
        - تعداد اتاق (rooms_count)
        - طبقه (floor)
        - پارکینگ (has_parking)
        - بالکن (has_balcony)
        - انبار (has_warehouse)
        - سال ساخت (construction_year)
        - نوع سند (deed_type)
        - تامین‌کننده آب گرم (has_warm_water_provider)
        - سیستم گرمایشی (has_heating_system)
        - سیستم سرمایشی (has_cooling_system)
        - نوع سرویس بهداشتی (has_restroom)
        - جهت ساختمان (building_direction)
        - جنس کف (floor_material)
        - تعداد کل طبقات (total_floors_count)
        - تعداد واحد در هر طبقه (unit_per_floor)
        - بازسازی شده یا نه (is_rebuilt)
        """
        print('لطفا اطلاعات زیر رو وارد کن تا پردازش کنم')
        done = False
        return answer, done
    else:
        # generating answer related to the user's sentence and asking for fields to fill
        print('api generated')
        done = False
        return answer, done

Extracted: {'city_slug': 'mashhad', 'rooms_count': 10, 'has_elevator': True}
Testing for: {'city_slug': 'mashhad', 'rooms_count': 10, 'has_elevator': True} 
------------------------------------------
invalid data found
- تعداد اتاق
- محله (neighborhood_slug)
- قیمت (price_value)
- زیربنا (متر مربع) (building_size)
- تعداد اتاق (rooms_count)
- طبقه (floor)
- پارکینگ (has_parking)
- بالکن (has_balcony)
- انبار (has_warehouse)
- سال ساخت (construction_year)
- نوع سند (deed_type)
- تامین‌کننده آب گرم (has_warm_water_provider)
- سیستم گرمایشی (has_heating_system)
- سیستم سرمایشی (has_cooling_system)
- نوع سرویس بهداشتی (has_restroom)
- جهت ساختمان (building_direction)
- جنس کف (floor_material)
- تعداد کل طبقات (total_floors_count)
- تعداد واحد در هر طبقه (unit_per_floor)
- بازسازی شده یا نه (is_rebuilt)
--- out of range ---
 ['rooms_count']
--- missing ---
 [('neighborhood_slug', 'محله'), ('price_value', 'قیمت'), ('building_size', 'زیربنا (متر مربع)'), ('rooms_count', 'تعداد اتاق'), ('floor

In [119]:
# ------------------------------
# Constants
# ------------------------------
FIELDS_ORDER = [
    "city_slug", "neighborhood_slug", "price_value", "building_size",
    "rooms_count", "floor", "has_parking", "has_elevator", "has_balcony",
    "has_warehouse", "construction_year", "deed_type", "has_warm_water_provider",
    "has_heating_system", "has_cooling_system", "has_restroom", "building_direction",
    "floor_material", "total_floors_count", "unit_per_floor", "is_rebuilt"
]

# sample mapping for Farsi field names (expand this yourself)
column_name_fa = {
    "city_slug": "شهر",
    "neighborhood_slug": "محله",
    "price_value": "قیمت",
    "building_size": "زیربنا",
    "rooms_count": "تعداد اتاق",
    "floor": "طبقه",
    "has_parking": "پارکینگ",
    "has_elevator": "آسانسور",
    "has_balcony": "بالکن",
    "has_warehouse": "انبار",
    "construction_year": "سال ساخت",
    "deed_type": "نوع سند",
    "has_warm_water_provider": "تامین‌کننده آب گرم",
    "has_heating_system": "سیستم گرمایشی",
    "has_cooling_system": "سیستم سرمایشی",
    "has_restroom": "نوع سرویس بهداشتی",
    "building_direction": "جهت ساختمان",
    "floor_material": "جنس کف",
    "total_floors_count": "تعداد کل طبقات",
    "unit_per_floor": "تعداد واحد در هر طبقه",
    "is_rebuilt": "بازسازی شده یا نه",
}

# ------------------------------
# Utilities
# ------------------------------
def check_json_fields(json_data, col_map=column_name_fa, min_required=4):
    """Check out-of-range + missing fields. Return structured info."""

    # --- Out of range detection ---
    out_of_range = [
        key for key, val in json_data.items() if val == "❌out of range"
    ]

    # --- Normalize min_/max_ ---
    normalized_fields = set()
    for key, val in json_data.items():
        if val not in (None, "", [], "❌out of range"):
            base_key = key.replace("min_", "").replace("max_", "")
            normalized_fields.add(base_key)

    num_filled = len(normalized_fields)

    # --- Missing in logical order ---
    missing = [
        key for key in FIELDS_ORDER if key not in normalized_fields
    ]

    return out_of_range, missing, num_filled


# ------------------------------
# Main pipeline
# ------------------------------
def process(user_text, request_count, df, min_required=4):
    """Process one user turn and decide what to do next."""

    done = False
    extracted = parse_user_query(user_text)
    print("Extracted:", extracted)

    validated_range = range_test(extracted)

    out_of_range, missing, num_filled = check_json_fields(validated_range)

    print("--- Out of range ---", out_of_range)
    print("--- Missing ---", missing)
    print("--- Number filled ---", num_filled)

    if num_filled >= min_required:
        print("✅ اطلاعات کافی داریم. در حال فیلتر کردن...")
        df_results = filter_dataframe(df, validated_range)
        answer = "🔎 پیدا کردن خانه با مشخصات گفته شده..."
        done = True
        return df_results, answer, done, request_count, None

    # not enough fields
    request_count += 1

    if request_count > 3:
        # give up and ask user for explicit fields
        to_print = "\n".join(
            f"- {column_name_fa.get(key, key)} ({key})" for key in FIELDS_ORDER
        )
        answer = f"⚠️ لطفا اطلاعات زیر رو وارد کن تا پردازش کنم:\n{to_print}"
        done = False
        return None, answer, done, request_count, user_text
    else:
        # ask for missing ones only
        needed = [
            f"- {column_name_fa.get(k, k)} ({k})" for k in missing[:5]
        ]
        answer = (
            extracted.get("answer", "لطفا اطلاعات بیشتری بده 🙏") +
            "\n\n👇 این موارد هم لازمه:\n" + "\n".join(needed)
        )
        done = False
        return None, answer, done, request_count, user_text


def session():
    request_count = 0
    user_text = "یک خونه در مشهد با 10 تا اتاق میخوام که آسانسور داشته باشه"
    done = False

    while not done:
        df_results, answer, done, request_count, prev_user_text = process(
            user_text, request_count, df
        )
            
        print("Bot:", answer)
        if done:
            print("🎉 جستجو کامل شد، نتایج آماده‌ست")
            print(df_results.head())
            return df_results
        # in real bot, get next input from user
        user_text = input("User: ")

        if prev_user_text:
            user_text = prev_user_text + ". " + user_text
            print("Combined:", user_text)

In [108]:
session()

Extracted: {'city_slug': 'mashhad', 'rooms_count': 10, 'has_elevator': True}
Testing for: {'city_slug': 'mashhad', 'rooms_count': 10, 'has_elevator': True} 
------------------------------------------
--- Out of range --- ['rooms_count']
--- Missing --- ['neighborhood_slug', 'price_value', 'building_size', 'rooms_count', 'floor', 'has_parking', 'has_balcony', 'has_warehouse', 'construction_year', 'deed_type', 'has_warm_water_provider', 'has_heating_system', 'has_cooling_system', 'has_restroom', 'building_direction', 'floor_material', 'total_floors_count', 'unit_per_floor', 'is_rebuilt']
--- Number filled --- 2
Bot: لطفا اطلاعات بیشتری بده 🙏

👇 این موارد هم لازمه:
- محله (neighborhood_slug)
- قیمت (price_value)
- زیربنا (building_size)
- تعداد اتاق (rooms_count)
- طبقه (floor)


User:  توی محله راهنمایی باشه


Combined: یک خونه در مشهد با 10 تا اتاق میخوام که آسانسور داشته باشه. توی محله راهنمایی باشه
Extracted: {'city_slug': 'mashhad', 'neighborhood_slug': 'rahnumayi', 'rooms_count': 10, 'has_elevator': True}
Testing for: {'city_slug': 'mashhad', 'neighborhood_slug': 'rahnumayi', 'rooms_count': 10, 'has_elevator': True} 
------------------------------------------
--- Out of range --- ['rooms_count']
--- Missing --- ['price_value', 'building_size', 'rooms_count', 'floor', 'has_parking', 'has_balcony', 'has_warehouse', 'construction_year', 'deed_type', 'has_warm_water_provider', 'has_heating_system', 'has_cooling_system', 'has_restroom', 'building_direction', 'floor_material', 'total_floors_count', 'unit_per_floor', 'is_rebuilt']
--- Number filled --- 3
Bot: لطفا اطلاعات بیشتری بده 🙏

👇 این موارد هم لازمه:
- قیمت (price_value)
- زیربنا (building_size)
- تعداد اتاق (rooms_count)
- طبقه (floor)
- پارکینگ (has_parking)


User:  خب پارکینگ هم نداشته باشه. برو بگرد


Combined: یک خونه در مشهد با 10 تا اتاق میخوام که آسانسور داشته باشه. توی محله راهنمایی باشه. خب پارکینگ هم نداشته باشه. برو بگرد
Extracted: {'city_slug': 'mashhad', 'neighborhood_slug': 'rahnamayi', 'rooms_count': 10, 'has_elevator': True, 'has_parking': False}
Testing for: {'city_slug': 'mashhad', 'neighborhood_slug': 'rahnamayi', 'rooms_count': 10, 'has_elevator': True, 'has_parking': False} 
------------------------------------------
--- Out of range --- ['rooms_count']
--- Missing --- ['price_value', 'building_size', 'rooms_count', 'floor', 'has_balcony', 'has_warehouse', 'construction_year', 'deed_type', 'has_warm_water_provider', 'has_heating_system', 'has_cooling_system', 'has_restroom', 'building_direction', 'floor_material', 'total_floors_count', 'unit_per_floor', 'is_rebuilt']
--- Number filled --- 4
✅ اطلاعات کافی داریم. در حال فیلتر کردن...
Bot: 🔎 پیدا کردن خانه با مشخصات گفته شده...
🎉 جستجو کامل شد، نتایج آماده‌ست
Empty DataFrame
Columns: [cat2_slug, cat3_slug, city_slug,

In [120]:
df_results = session()

Extracted: {'city_slug': 'mashhad', 'rooms_count': 10, 'has_elevator': True}
Testing for: {'city_slug': 'mashhad', 'rooms_count': 10, 'has_elevator': True} 
------------------------------------------
--- Out of range --- ['rooms_count']
--- Missing --- ['neighborhood_slug', 'price_value', 'building_size', 'rooms_count', 'floor', 'has_parking', 'has_balcony', 'has_warehouse', 'construction_year', 'deed_type', 'has_warm_water_provider', 'has_heating_system', 'has_cooling_system', 'has_restroom', 'building_direction', 'floor_material', 'total_floors_count', 'unit_per_floor', 'is_rebuilt']
--- Number filled --- 2
Bot: لطفا اطلاعات بیشتری بده 🙏

👇 این موارد هم لازمه:
- محله (neighborhood_slug)
- قیمت (price_value)
- زیربنا (building_size)
- تعداد اتاق (rooms_count)
- طبقه (floor)


User:  توی محله راهنمایی باشه و پارکینگ داشته باشه


Combined: یک خونه در مشهد با 10 تا اتاق میخوام که آسانسور داشته باشه. توی محله راهنمایی باشه و پارکینگ داشته باشه
Extracted: {'city_slug': 'mashhad', 'neighborhood_slug': 'rahnamayi', 'rooms_count': 10, 'has_elevator': True, 'has_parking': True}
Testing for: {'city_slug': 'mashhad', 'neighborhood_slug': 'rahnamayi', 'rooms_count': 10, 'has_elevator': True, 'has_parking': True} 
------------------------------------------
--- Out of range --- ['rooms_count']
--- Missing --- ['price_value', 'building_size', 'rooms_count', 'floor', 'has_balcony', 'has_warehouse', 'construction_year', 'deed_type', 'has_warm_water_provider', 'has_heating_system', 'has_cooling_system', 'has_restroom', 'building_direction', 'floor_material', 'total_floors_count', 'unit_per_floor', 'is_rebuilt']
--- Number filled --- 4
✅ اطلاعات کافی داریم. در حال فیلتر کردن...
Bot: 🔎 پیدا کردن خانه با مشخصات گفته شده...
🎉 جستجو کامل شد، نتایج آماده‌ست
              cat2_slug       cat3_slug city_slug neighborhood_slug  \
2401 

In [122]:
df_results

,cat2_slug,cat3_slug,city_slug,neighborhood_slug,user_type,description,title,rent_mode,price_mode,price_value,...,has_heating_system,has_cooling_system,has_restroom,building_direction,floor_material,location_latitude,location_longitude,location_radius,year_created,month_created
2401,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,آپارتمان ۱۰۷ متر. ۳متر انباری\nطبقه دوم\nآسانس...,آپارتمان ۱۱۰متر با آسانسور و پارکینگ,NaN,مقطوع,4.900000e+09,...,NaN,NaN,NaN,north,NaN,36.304173,59.575726,500.0,2024,6
18766,residential-sell,apartment-sell,mashhad,rahnamaei,مشاور املاک,☑️ مشاور شما : اشراقی ☑️\n☑️ راهنمایی 6 مسکن ب...,آپارتمان صفر ابتدا راهنمایی سازند برند,NaN,مقطوع,8.500000e+09,...,shoofaj,air_conditioner,squat_seat,south,ceramic,36.299683,59.569866,500.0,2024,8
22815,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,۲۶۰متر ۳خواب دو مستر طبقه چهارم تک واحدی \nدو...,اپارتمان۲۶۰متر بخارایی,NaN,مقطوع,1.600000e+10,...,shoofaj,air_conditioner,seat,north,ceramic,36.302433,59.574223,NaN,2024,8
32658,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,----------با درود و احترام ---------\n\n✅یک وا...,آپارتمان ۱۱۲ متر بخارایی سلمان فارسی,NaN,مقطوع,5.600000e+09,...,shoofaj,water_cooler,squat,south,ceramic,36.301891,59.572247,NaN,2024,11
68413,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,خدایا شکرت♥️\n\nبا سلام و احترام\n____\nفروش ی...,(۱۲۵ متر راهنمایی )دو نبش/صفر/ نواب,NaN,مقطوع,9.150000e+09,...,NaN,NaN,NaN,NaN,NaN,36.303104,59.571140,0.0,2024,12
82950,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,آپارتمان ۱۷۲متری طبقه ۲\nکف سرامیک کابینت های ...,۱۷۲متری معاوضه راهنمایی,NaN,مقطوع,1.080000e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,11
96838,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,✅۳ خواب/یک خواب مستر\n✅حال پذیرایی بزرگ غرق در...,۱۶۰متر/تک واحدی/خوش نقشه/راهنمایی,NaN,مقطوع,1.000000e+10,...,NaN,NaN,NaN,NaN,NaN,36.299736,59.569500,NaN,2024,12
103289,residential-sell,apartment-sell,mashhad,rahnamaei,مشاور املاک,۶۵ متری \nطبقه دوم رو به فسیلیتی \nبلوک ۴ فاز ...,آپارتمان ۶۵متر درقبرس شمالی,NaN,مقطوع,3.100000e+09,...,split,split,seat,north,ceramic,36.301369,59.572205,500.0,2024,7
110631,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,سند۶دانگ…اسانسور…پارکینگ رمپ دار…تک واحدی…\nکا...,اپارتمان ۲۵۰متری سازمان اب((توفیق)),NaN,مقطوع,1.800000e+10,...,shoofaj,air_conditioner,squat_seat,north,wood_parquet,NaN,NaN,NaN,2024,10
110650,residential-sell,apartment-sell,mashhad,rahnamaei,مشاور املاک,کف سرامیک کابینت ام دی اف پکیج پارکینگ انباری ...,بخارایی ۲۴۰متر ۷سال ساخت ط اول با آسانسور,NaN,مقطوع,1.400000e+10,...,shoofaj,split,squat_seat,north,ceramic,36.302212,59.573162,500.0,2024,6


**Validating** by search (CORRECT)

In [113]:
df[(df['city_slug'] == 'mashhad') & (df['neighborhood_slug'] == 'rahnamaei') & (df['has_elevator'] == True) & (df['has_parking'] == True)]

,cat2_slug,cat3_slug,city_slug,neighborhood_slug,user_type,description,title,rent_mode,price_mode,price_value,...,has_heating_system,has_cooling_system,has_restroom,building_direction,floor_material,location_latitude,location_longitude,location_radius,year_created,month_created
2401,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,آپارتمان ۱۰۷ متر. ۳متر انباری\nطبقه دوم\nآسانس...,آپارتمان ۱۱۰متر با آسانسور و پارکینگ,NaN,مقطوع,4.900000e+09,...,NaN,NaN,NaN,north,NaN,36.304173,59.575726,500.0,2024,6
18766,residential-sell,apartment-sell,mashhad,rahnamaei,مشاور املاک,☑️ مشاور شما : اشراقی ☑️\n☑️ راهنمایی 6 مسکن ب...,آپارتمان صفر ابتدا راهنمایی سازند برند,NaN,مقطوع,8.500000e+09,...,shoofaj,air_conditioner,squat_seat,south,ceramic,36.299683,59.569866,500.0,2024,8
22815,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,۲۶۰متر ۳خواب دو مستر طبقه چهارم تک واحدی \nدو...,اپارتمان۲۶۰متر بخارایی,NaN,مقطوع,1.600000e+10,...,shoofaj,air_conditioner,seat,north,ceramic,36.302433,59.574223,NaN,2024,8
32658,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,----------با درود و احترام ---------\n\n✅یک وا...,آپارتمان ۱۱۲ متر بخارایی سلمان فارسی,NaN,مقطوع,5.600000e+09,...,shoofaj,water_cooler,squat,south,ceramic,36.301891,59.572247,NaN,2024,11
68413,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,خدایا شکرت♥️\n\nبا سلام و احترام\n____\nفروش ی...,(۱۲۵ متر راهنمایی )دو نبش/صفر/ نواب,NaN,مقطوع,9.150000e+09,...,NaN,NaN,NaN,NaN,NaN,36.303104,59.571140,0.0,2024,12
82950,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,آپارتمان ۱۷۲متری طبقه ۲\nکف سرامیک کابینت های ...,۱۷۲متری معاوضه راهنمایی,NaN,مقطوع,1.080000e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,11
96838,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,✅۳ خواب/یک خواب مستر\n✅حال پذیرایی بزرگ غرق در...,۱۶۰متر/تک واحدی/خوش نقشه/راهنمایی,NaN,مقطوع,1.000000e+10,...,NaN,NaN,NaN,NaN,NaN,36.299736,59.569500,NaN,2024,12
103289,residential-sell,apartment-sell,mashhad,rahnamaei,مشاور املاک,۶۵ متری \nطبقه دوم رو به فسیلیتی \nبلوک ۴ فاز ...,آپارتمان ۶۵متر درقبرس شمالی,NaN,مقطوع,3.100000e+09,...,split,split,seat,north,ceramic,36.301369,59.572205,500.0,2024,7
110631,residential-sell,apartment-sell,mashhad,rahnamaei,NaN,سند۶دانگ…اسانسور…پارکینگ رمپ دار…تک واحدی…\nکا...,اپارتمان ۲۵۰متری سازمان اب((توفیق)),NaN,مقطوع,1.800000e+10,...,shoofaj,air_conditioner,squat_seat,north,wood_parquet,NaN,NaN,NaN,2024,10
110650,residential-sell,apartment-sell,mashhad,rahnamaei,مشاور املاک,کف سرامیک کابینت ام دی اف پکیج پارکینگ انباری ...,بخارایی ۲۴۰متر ۷سال ساخت ط اول با آسانسور,NaN,مقطوع,1.400000e+10,...,shoofaj,split,squat_seat,north,ceramic,36.302212,59.573162,500.0,2024,6


## Working on identifying similarities between `neighborhood_slug` of LLM and actual

In [2]:
df = pd.read_csv('filtered_df_cleaned_v2.csv')
df.info()

C:\Users\Parinaz\AppData\Local\Temp\ipykernel_10452\2775316985.py:1: DtypeWarning: Columns (18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('filtered_df_cleaned_v2.csv')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303385 entries, 0 to 303384
Data columns (total 34 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   cat2_slug                303385 non-null  object 
 1   cat3_slug                303385 non-null  object 
 2   city_slug                303385 non-null  object 
 3   neighborhood_slug        167946 non-null  object 
 4   user_type                110664 non-null  object 
 5   description              303385 non-null  object 
 6   title                    303372 non-null  object 
 7   rent_mode                0 non-null       float64
 8   price_mode               303383 non-null  object 
 9   price_value              303380 non-null  float64
 10  credit_mode              0 non-null       float64
 11  building_size            303380 non-null  float64
 12  deed_type                185556 non-null  object 
 13  floor                    303377 non-null  float64
 14  room

In [3]:
df_m = df[df['city_slug'] == 'mashhad']
df_m.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24331 entries, 1 to 303375
Data columns (total 34 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   cat2_slug                24331 non-null  object 
 1   cat3_slug                24331 non-null  object 
 2   city_slug                24331 non-null  object 
 3   neighborhood_slug        24331 non-null  object 
 4   user_type                8983 non-null   object 
 5   description              24331 non-null  object 
 6   title                    24325 non-null  object 
 7   rent_mode                0 non-null      float64
 8   price_mode               24331 non-null  object 
 9   price_value              24331 non-null  float64
 10  credit_mode              0 non-null      float64
 11  building_size            24331 non-null  float64
 12  deed_type                15315 non-null  object 
 13  floor                    24331 non-null  float64
 14  rooms_count              2

In [4]:
df_m.neighborhood_slug.value_counts()

neighborhood_slug
elahiyehblvd        3924
ghasemabad          2286
bolvartoos          1249
azadshahr           1045
north-tabars         794
                    ... 
kuy-e-police           2
dehnavi                2
Shahrak-e-Shirin       2
mooud                  1
neyzeh                 1
Name: count, Length: 147, dtype: int64

In [5]:
df_m.neighborhood_slug.unique()

array(['emamreza', 'faramarzabbasi', 'farhang', 'ghasemabad',
       'honarestan', 'elahiyehblvd', 'north-tabars', 'haftetir',
       'bolvartoos', 'north-shahid-mottahari', 'ayatollah-ebadi',
       'hoseynabad', 'ayatollah-khamenei', 'daneshjoo', 'eqbal',
       'sayyadshirazi', 'vakilabad', 'rezashahr', 'sadjadshahr',
       'sanabad', 'hashemieh', 'Kuy-e-Seyyedi', 'balal',
       'amiriyeh-mashhad', 'hojjat', 'kuy-e-police', 'kuy-e-raddeh',
       'aboutaleb', 'resalat-mashhad', 'janbaz', 'azadshahr', 'ahmadabad',
       'tollab', 'kuy-e-sarafrazan', 'abdolmotalleb', 'fareqoltahsilan',
       'mousavighoochani', 'khaje-rabi', 'jahedshahr', 'isargaran',
       'pardis-mashhad', 'southmotahhari', 'shahid-honarvar',
       'ferdowsi-university-of-mashhad', 'baharestan-mashhad', '10-day',
       'samen', 'panj-tan-al-e-abba', 'telgerd', 'emamkhomeini',
       'kuy-e-mahdi', 'rahnamaei', 'barqsquare', 'eastgaz', 'samarghand',
       'sadaf', 'abkuh', 'kuy-e-kowsar', 'shahrara-mashhad',


In [10]:
candidates = ["mashhad-hashemi-nejad-international-airport", "malekabad",
              "rahnamaieh", "palestin"]
query = "international_airport"
best_match, score, idx = process.extractOne(query, candidates, scorer=fuzz.ratio)
print(best_match, score) # I think 0.6 is good threshold

mashhad-hashemi-nejad-international-airport 62.5


In [14]:
query = "airport"

candidates = df_m['neighborhood_slug'].dropna().astype(str).str.lower().tolist()
query_lower = query.lower()

# 1️⃣ Substring match first
substring_matches = [c for c in candidates if query_lower in c or c in query_lower]

if substring_matches:
    best_match = min(substring_matches, key=len)  # shortest match
else:
    # 2️⃣ Fuzzy token match using RapidFuzz
    best_match, score, idx = process.extractOne(query_lower, candidates, scorer=fuzz.token_set_ratio)
    if score < 70:
        best_match = None

if best_match:
    print(f"Best match for '{query}': {best_match}")
    matched_rows = df_m[df_m['neighborhood_slug'].str.lower() == best_match]
else:
    print(f"No match found for '{query}'")
    matched_rows = pd.DataFrame(columns=df_m.columns)

matched_rows

Best match for 'airport': mashhad-hashemi-nejad-international-airport


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,user_type,description,title,rent_mode,price_mode,price_value,...,has_heating_system,has_cooling_system,has_restroom,building_direction,floor_material,location_latitude,location_longitude,location_radius,year_created,month_created
61322,residential-sell,apartment-sell,mashhad,mashhad-hashemi-nejad-international-airport,NaN,آپارتمان زیربنا 75 بنا 55 متر مفید\nاتاق خواب...,اپارتمان یک خوابه با حیاط مشترک,NaN,مقطوع,1.850000e+09,...,heater,water_cooler,squat,north,ceramic,36.232353,59.630257,NaN,2024,9
104326,residential-sell,apartment-sell,mashhad,mashhad-hashemi-nejad-international-airport,NaN,واقع در شهرک عسگریه...\nطبقه اول...\nکناف پردا...,* آپارتمان ۷۵ متری/خوش نقشه *,NaN,مقطوع,1.600000e+09,...,NaN,NaN,NaN,NaN,NaN,36.238457,59.642590,NaN,2024,12


# Previous exploration

In [2]:
df = pd.read_csv('filled_data_v0.csv')
df.head()

,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,price_value,land_size,...,has_jacuzzi,has_sauna,floor_material,property_type,location_latitude,location_longitude,location_radius,size_category,year_category,rooms_count_numeric
0,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,8.500000e+09,NaN,...,False,False,unselect,other,NaN,NaN,500.0,50-100m²,NaN,1
1,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01 00:00:00,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,5.750000e+09,NaN,...,False,False,ceramic,other,NaN,NaN,NaN,100-150m²,NaN,2
2,residential-sell,apartment-sell,tehran,dardasht,2024-09-01 00:00:00,مشاور املاک,♡♡♡♡♡♡بنام خدا♡♡♡♡♡♡\n♡♡♡عرض ادب واحترام♡♡♡\n♡...,اپارتمان ۱۰۰متری طبقه۳و۴ فول دردشت شمالی,8.700000e+09,NaN,...,False,False,unselect,other,35.729832,51.505466,500.0,50-100m²,NaN,2
3,residential-sell,apartment-sell,mahdasht-city,NaN,2024-06-01 00:00:00,مشاور املاک,با سلام\n\nاملاک بزرگ اتحاد با ۲شعبه فعال با ف...,۷۸متری دوبحرنورگیر بازسازی شده شیک سندتک برگ,6.500000e+08,NaN,...,False,False,ceramic,other,35.712364,50.794781,500.0,50-100m²,NaN,2
4,residential-sell,apartment-sell,mashhad,faramarzabbasi,2024-10-01 00:00:00,شخصی,اول رسالت فرد نزدیک به حاشیه فرامرز .. ساکنین...,متراژ ۸۰ متر فرامرز نزدیک حاشیه,3.000000e+09,NaN,...,False,False,ceramic,other,NaN,NaN,NaN,50-100m²,NaN,2


**Selecting relatable columns for processing user input:**

**THE COLUMNS:**

**The main ones** (from my pov):
- `cat3_slug`
- `city_slug`
- `neighborhood_slug`
- `price_value` (*range*)
- `land_size` (*range*)
- `buiding_size` (*range*)
- `rooms_count`

**Sure to put:** `cat2_slug`, 'cat3_slug', 'city_slug', 'neighborhood_slug', 'price_value' (*range*), 'land_size' (*range*), 'buiding_size' (*range*), 'rooms_count', `has_balcony`, `has_elevator`, `has_warehouse`, `has_parking`, `has_water`, `has_warm_water_provider`, `has_electricity`, `has_gas`, `has_heating_system`, `has_cooling system`, `has_restroom`, `has_security_guard`, `has_barbecue`, `building_direction` (**maybe good**), `has_pool`, `has_jacuzzi`, `has_sauna`, `floor_material`, `property_type` (beach, jungle or ...)

**Can be put** but **additional** (The user *may* specify): `created_at_month` which holds the year and month., `deed_type` (written agreement or ...), `has_business_deed`, `total_floors_count` (**???** what is it exactly), `unit_per_floor`, `is_rebuild`

**Sure** to **not** put:
- `user_type` (شخصی or مشاور املاک doesn't matter)
- `description`
- `title`
- `floor` -> It has numbers and we don't know what material is for each number
- `location_latitude` (**may still use for geo**) -> It's not like a user is going to specify latitude and longitude
- `location_radius` (**may still use for geo**)

**Note1**: In the `title` there are also descriptions where it could help filling the NaN values besides `description`

**Note2** (*addressed*): field `construction_year` and `year_category` is all NaN

**Note3** (*addressed*): In this dataset a lot of `NaN` values were filled with `'unselect'` So I should first turn them into NaN to check

In [30]:
df.to_csv('filled_data_v1.csv', index=False)

# Filtering the data on `apartment-sell` and `residential-sell`

Using `residential-sell` from cat2_slug and `apartment-sell` from cat3_slug 

In [21]:
df = pd.read_csv('filled_data_v1.csv', low_memory=False)

In [23]:
filtered_df = df[
    (df['cat2_slug'] == 'residential-sell') &
    (df['cat3_slug'] == 'apartment-sell')
]

# User Input

In [29]:
df = filtered_df

In [30]:
df.head()

,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,price_value,land_size,...,has_pool,has_jacuzzi,has_sauna,floor_material,property_type,location_latitude,location_longitude,location_radius,size_category,rooms_count_numeric
0,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,8.500000e+09,NaN,...,False,False,False,NaN,other,NaN,NaN,500.0,50-100m²,1
1,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01 00:00:00,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,5.750000e+09,NaN,...,False,False,False,ceramic,other,NaN,NaN,NaN,100-150m²,2
2,residential-sell,apartment-sell,tehran,dardasht,2024-09-01 00:00:00,مشاور املاک,♡♡♡♡♡♡بنام خدا♡♡♡♡♡♡\n♡♡♡عرض ادب واحترام♡♡♡\n♡...,اپارتمان ۱۰۰متری طبقه۳و۴ فول دردشت شمالی,8.700000e+09,NaN,...,False,False,False,NaN,other,35.729832,51.505466,500.0,50-100m²,2
3,residential-sell,apartment-sell,mahdasht-city,NaN,2024-06-01 00:00:00,مشاور املاک,با سلام\n\nاملاک بزرگ اتحاد با ۲شعبه فعال با ف...,۷۸متری دوبحرنورگیر بازسازی شده شیک سندتک برگ,6.500000e+08,NaN,...,False,False,False,ceramic,other,35.712364,50.794781,500.0,50-100m²,2
4,residential-sell,apartment-sell,mashhad,faramarzabbasi,2024-10-01 00:00:00,شخصی,اول رسالت فرد نزدیک به حاشیه فرامرز .. ساکنین...,متراژ ۸۰ متر فرامرز نزدیک حاشیه,3.000000e+09,NaN,...,False,False,False,ceramic,other,NaN,NaN,NaN,50-100m²,2


In [31]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
)

COLUMNS = df.columns

COLUMNS

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
       'created_at_month', 'user_type', 'description', 'title', 'price_value',
       'land_size', 'building_size', 'deed_type', 'has_business_deed', 'floor',
       'rooms_count', 'total_floors_count', 'unit_per_floor', 'has_balcony',
       'has_elevator', 'has_warehouse', 'has_parking', 'is_rebuilt',
       'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas',
       'has_heating_system', 'has_cooling_system', 'has_restroom',
       'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool',
       'has_jacuzzi', 'has_sauna', 'floor_material', 'property_type',
       'location_latitude', 'location_longitude', 'location_radius',
       'size_category', 'rooms_count_numeric'],
      dtype='object')

## Extracting columns from a user input text

In [41]:
long_des = df.loc[df['description'].str.len() > 500]
print(len(long_des))
long_des.head()

71569


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,price_value,land_size,...,has_pool,has_jacuzzi,has_sauna,floor_material,property_type,location_latitude,location_longitude,location_radius,size_category,rooms_count_numeric
3,residential-sell,apartment-sell,mahdasht-city,NaN,2024-06-01 00:00:00,مشاور املاک,با سلام\n\nاملاک بزرگ اتحاد با ۲شعبه فعال با ف...,۷۸متری دوبحرنورگیر بازسازی شده شیک سندتک برگ,6.500000e+08,NaN,...,False,False,False,ceramic,other,35.712364,50.794781,500.0,50-100m²,2
6,residential-sell,apartment-sell,mashhad,farhang,2024-06-01 00:00:00,شخصی,با سلام\n\n✨✨ شما لایق بهترین ها هستید ✨✨\n\n✅...,تاپ لوکیشن جلال سروش ۱۶۹متر کلید موجود قبل فرهنگ,1.299000e+10,NaN,...,False,False,False,ceramic,other,36.321831,59.524502,500.0,150-200m²,3
16,residential-sell,apartment-sell,arak,NaN,2024-10-01 00:00:00,شخصی,⚛ از خاک تا ساخت با شما هستیم ⚛\n\n✴ واحدی زیب...,فروش آپارتمان/۱۱۴متر/مصطفی خمینی/سمت پارک,5.700000e+09,NaN,...,False,False,False,ceramic,other,NaN,NaN,NaN,100-150m²,2
18,residential-sell,apartment-sell,tabriz,NaN,2024-11-01 00:00:00,شخصی,•●♥ ﷽ ♥●•\n\n⚜️کـارشنـاس ارشــد امور ملکی سهند...,فاز ۳ سهند بنفشه,1.120000e+09,NaN,...,False,False,False,wood_parquet,other,NaN,NaN,0.0,50-100m²,2
25,residential-sell,apartment-sell,mashhad,ghasemabad,2024-06-01 00:00:00,مشاور املاک,▪️▫️⚜بنام خداوند روزی دهنده مهربان⚜▫️▪️ \n\n\n...,آپارتمان ۱۱۰متری*خوش چیدمان*۸واحدی/ فلاحی۳۸,3.500000e+09,NaN,...,False,False,False,ceramic,other,36.365303,59.499275,500.0,100-150m²,2


In [43]:
print(long_des.iloc[0]['description'])

با سلام

املاک بزرگ اتحاد با ۲شعبه فعال با فایلینگ قوی در منطقه...
بهترین آپارتمان های مناسب و به قیمت منطقه را از ما بخواهید...
 
⭕️حس خوب خرید ملک را با ما تجربه کنید⭕️

(واحدها واقع در ماهدشت می باشد)

✅️ ۷۸متر
✅️روبه نما غرق نور 
✅️پارکینگ اختصاصی 
✅️آسانسور،انباری
✅️سازه مهندسی برند منطقه 
✅️سند تک برگ
✅️لوکیشن تا مرکز خرید و تاکسی ۲دقیقه

شرایط پرداخت⁉️
نقدینگی اولیه شما مبلغ آگهی میباشد+۶۴۰ میلیون وام خرید مسکن+۳۰۰ میلیون رهن مستاجر=قیمت کل❗️❗️❗️
❌️مبلغ ذکر شده نقدینگی شماست،رهن و وام از مبلغ کل کسر شده❌️

مشاور و کارشناس امور ملکی شما:حسین کریمی


In [46]:
print(long_des.iloc[20]['description'])

فروش استثنایی: این آپارتمان با قیمت رقابتی و موقعیت عالی، فرصتی بی‌نظیر برای خرید است. از امکانات محلی و دسترسی آسان به شریان‌های اصلی شهر بهره‌مند شوید.

پروژه فاز ۸ :
متراژ : ۱۰۵ متر
جهت : شرقی
تعداد اتاق : دو خواب
وضعیت واحد : اولویت تحویل (نازک کاری)

چرا دپارتمان آرتین؟

 امنیت معامله: دپارتمان آرتین با داشتن پروانه کسب معتبر، اطمینان از انجام معاملات امن و مطمئن را به شما می‌دهد.

 خدمات بی‌واسطه: تمرکز ما بر ارائه خدمات منحصر به فرد و مستقیم به مشتریان است.

 تضمین کیفیت: تمامی پروژه‌های ما با بالاترین سطح کیفی و کامل‌ترین خدمات ارائه می‌شوند.

 همکاری اختصاصی: خدمات ما کاملاً اختصاصی بوده و با املاک‌های دیگر همکاری نمی‌کنیم، تا بتوانیم بهترین خدمات را به شما ارائه دهیم.

---

تماس و مشاوره:

من پوریا، مشاور تخصصی فاز ۸، آماده‌ام تا از ساعت ۹ صبح تا ۱۱ شب با شما در تماس باشم و مشاوره رایگان ارائه دهم. همین حالا تماس بگیرید و در مسیر موفقیت همراه شوید!


In [13]:
cond = (
    (df['neighborhood_slug'] == 'ونک') |
    (df['cat2_slug'] == 'ونک') |
    (df['cat3_slug'] == 'ونک') |
    (df['city_slug'] == 'ونک')
)
result = df[cond]

result

,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,price_value,land_size,...,has_jacuzzi,has_sauna,floor_material,property_type,location_latitude,location_longitude,location_radius,size_category,year_category,rooms_count_numeric


In [14]:
df['neighborhood_slug'].value_counts()

neighborhood_slug
elahiyehblvd                4654
poonak                      2870
bolvartoos                  2813
ghasemabad                  2661
jeyhoun                     2243
                            ... 
pardesar                       1
postak                         1
parastar-bandar-e-anzali       1
amir-shahid                    1
jamaran-rasht                  1
Name: count, Length: 1175, dtype: int64

In [17]:
df[df['neighborhood_slug'] == 'vanak']

,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,price_value,land_size,...,has_jacuzzi,has_sauna,floor_material,property_type,location_latitude,location_longitude,location_radius,size_category,year_category,rooms_count_numeric
607,residential-sell,apartment-sell,tehran,vanak,2024-06-01 00:00:00,مشاور املاک,تکواحدی \nبسیار خوش نقشه \n۱۲ متر انباری \nآفت...,فروش آپارتمان ۱۱۰ متری ونک شیخ بهایی شمالی,1.400000e+10,NaN,...,False,False,unselect,other,NaN,NaN,500.0,100-150m²,NaN,2
4127,residential-sell,house-villa-sell,tehran,vanak,2024-10-01 00:00:00,شخصی,فاصله یکساعتی \n\nمناسب سکونت دائمی \n\nاطلاعا...,فروش ویلایی,3.398000e+09,171.0,...,False,False,unselect,other,35.756561,51.410088,500.0,50-100m²,NaN,2
4690,residential-sell,apartment-sell,tehran,vanak,2024-07-01 00:00:00,مشاور املاک,تک واحد. باکلیه امکانات\n\nپلان بدون نقص \nخوا...,ونک.شیراز۲۰۵متر۳خ (فول بازسازی تکواحد),2.450000e+10,NaN,...,False,False,unselect,other,35.753094,51.398701,500.0,200-300m²,NaN,3
7836,residential-sell,apartment-sell,tehran,vanak,2024-11-01 00:00:00,شخصی,﷽ \n ...,۶۱متر۲خواب/پر‌باز‌ده فول/تو‌‌ضیحات بخو‌ن,3.100000e+09,NaN,...,False,False,ceramic,other,35.757721,51.400204,0.0,50-100m²,NaN,2
8475,residential-sell,apartment-sell,tehran,vanak,2024-10-01 00:00:00,شخصی,توضیحات:\n (((قابل توجه خریداران خوش پ...,آپارتمان/۱۴۵متر/۳ خ/ملاصدرا/کلیدنخورده,2.270000e+10,NaN,...,False,False,stone,other,35.755219,51.391861,500.0,100-150m²,NaN,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559453,residential-sell,plot-old,tehran,vanak,2024-12-01 00:00:00,شخصی,◾۳۲۰متر کلنگی \n◾۴۲۰متر بنا در دوطبقه \n◾پهنه ...,۳۲۰مترکلنگی۴۲۰متربنا در دوطبقه پهنهRتک‌مالک,9.500000e+10,NaN,...,False,False,unselect,other,NaN,NaN,NaN,300-500m²,NaN,2
560935,residential-sell,apartment-sell,tehran,vanak,2024-11-01 00:00:00,شخصی,✅در تلاشیم برای تحقق رویاهای شما✅\n\n✔۹۰۴ متر ...,۹۰۴ منر آپارتمان/مسکونی/موقعیت اداری/بصورت یکجا,2.440800e+11,NaN,...,False,False,stone,other,35.756542,51.396282,500.0,500-1000m²,NaN,5
565028,residential-sell,plot-old,tehran,vanak,2024-05-01 00:00:00,شخصی,تذکر، اگهی های کاذب و جعلی با عکس های کاذب که ...,فروش و معاوضه زمین متل قو شهرک رایکا,2.300000e+10,NaN,...,False,False,unselect,other,35.757816,51.409565,NaN,300-500m²,NaN,2
565460,commercial-sell,office-sell,tehran,vanak,2024-06-01 00:00:00,مشاور املاک,در مجتمع اداری خوشنام و برند\nطراحی و معماری م...,سند اداری، مدرن و مینیمال ۲۲۰ متر، هوشمند,7.700000e+10,NaN,...,False,False,unselect,other,NaN,NaN,500.0,200-300m²,NaN,2


**PROMPT Note**: prompt to add: if they entered a neighborhood like ونک, it should be converted to It's English form 'vanak' for query

and with `tehran`! not `تهران`

In [23]:
df[df['city_slug'] == 'tehran']

,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,price_value,land_size,...,has_jacuzzi,has_sauna,floor_material,property_type,location_latitude,location_longitude,location_radius,size_category,year_category,rooms_count_numeric
0,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,8.500000e+09,NaN,...,False,False,unselect,other,NaN,NaN,500.0,50-100m²,NaN,1
2,residential-sell,apartment-sell,tehran,dardasht,2024-09-01 00:00:00,مشاور املاک,♡♡♡♡♡♡بنام خدا♡♡♡♡♡♡\n♡♡♡عرض ادب واحترام♡♡♡\n♡...,اپارتمان ۱۰۰متری طبقه۳و۴ فول دردشت شمالی,8.700000e+09,NaN,...,False,False,unselect,other,35.729832,51.505466,500.0,50-100m²,NaN,2
9,residential-sell,plot-old,tehran,shahrak-naft-district5,2024-07-01 00:00:00,شخصی,۷۰۰ متر باغ ۱۰۰ ساله،\nواقع در روستای زیبای رن...,تهاتر یا فروش باغی زیبا در روستای زیبای رندان,1.800000e+10,NaN,...,False,False,unselect,other,35.890186,51.280197,NaN,500-1000m²,NaN,2
13,residential-sell,apartment-sell,tehran,kamraniyeh,2024-11-01 00:00:00,شخصی,سازه‌ایی‌بااصالت‌همجوارسفارت\nدربهترین‌فرعی‌بر...,کامرانیه‌شمالی‌ویوابدی۴۰۰متر۳جهت‌نور,1.330000e+11,NaN,...,False,False,unselect,other,NaN,NaN,NaN,300-500m²,NaN,4
23,residential-sell,apartment-sell,tehran,tohid,2024-10-01 00:00:00,شخصی,این واحد بدلیل نیاز مالی زیر قیمت بفروش میرسد ...,۱۱۰ متر * کلید نخورده * ستارخان,1.175000e+10,NaN,...,False,False,stone,other,NaN,NaN,NaN,100-150m²,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
566413,residential-sell,apartment-sell,tehran,masoudieh,2024-08-01 00:00:00,مشاور املاک,.. ☀️ .. ما ”عـمـارت“ هستیم\n ...,آپارتمان73m/نصبیات*1403*پارکینگ*انباری,3.950000e+09,NaN,...,False,False,ceramic,other,NaN,NaN,500.0,50-100m²,NaN,2
566426,residential-sell,apartment-sell,tehran,gomrok,2024-11-01 00:00:00,شخصی,۴۸متری انباری داره پارکینگ نداره طبقه چهارم نم...,آپارتمان ۴۸متری,2.480000e+09,NaN,...,False,False,ceramic,other,35.665260,51.384941,NaN,<50m²,NaN,1
566429,residential-sell,apartment-sell,tehran,gisha,2024-05-01 00:00:00,مشاور املاک,⚜️نقشه بی نظیر با سالن تفکیک شده\n⚜️ساختمان شم...,گیشا ۱۶۴ متر ۳ خواب تکواحدی,2.050000e+10,NaN,...,False,False,stone,other,35.728760,51.378708,500.0,150-200m²,NaN,3
566434,residential-sell,apartment-sell,tehran,abouzar,2024-05-01 00:00:00,مشاور املاک,⚜️⚜️⚜️املاک خدابنده ⚜️⚜️⚜️\n\n☎️مشاور منطقه نی...,آپارتمان ۸۰ متری /فروشنده واقعی/ نیروهوایی,6.800000e+09,NaN,...,False,False,stone,other,35.649338,51.478817,500.0,50-100m²,NaN,2


In [26]:
df.columns

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
       'created_at_month', 'user_type', 'description', 'title', 'price_value',
       'land_size', 'building_size', 'deed_type', 'has_business_deed', 'floor',
       'rooms_count', 'total_floors_count', 'unit_per_floor', 'has_balcony',
       'has_elevator', 'has_warehouse', 'has_parking', 'construction_year',
       'is_rebuilt', 'has_water', 'has_warm_water_provider', 'has_electricity',
       'has_gas', 'has_heating_system', 'has_cooling_system', 'has_restroom',
       'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool',
       'has_jacuzzi', 'has_sauna', 'floor_material', 'property_type',
       'location_latitude', 'location_longitude', 'location_radius',
       'size_category', 'year_category', 'rooms_count_numeric'],
      dtype='object')

In [27]:
df['building_size']

0          60.0
1         115.0
2         100.0
3          78.0
4          80.0
          ...  
566439    140.0
566440     91.0
566441    105.0
566442    180.0
566443    200.0
Name: building_size, Length: 566444, dtype: float64

**PROMPT Note**: In some cases like `building_size` it can either be a *range* or *fixed vlaue*. That prompt for handling this and retreving the suitable rows should also be added.

**PROMPT Note**: Also, in the example that I gave the model, دفتر اداری is something which is handled in one of the columns (I think `cat3_slug` or sth else) but the LLM didn't recognize because it lacked to see the **values of each column**. For each column is should know the values (if they're not much, like 10 values) or a range (for example if there exist a lot of individual values for something like `size`, it should know a range and *format* of it) 